# Comprehensive Pandas Masterclass
## From First Principles to Advanced Analytics

**Datasets used in this notebook**
- `sales_transactions.csv` — 510 rows, 13 columns, intentional missing values, duplicates, and mixed types
- `pandas_training_data.xlsx` — 5 sheets: Employees, Products, Monthly_Sales, Customer_Feedback, Budget_vs_Actuals

**Topics covered**
1. Setup & Pandas fundamentals  
2. Data structures — Series & DataFrame in depth  
3. Reading data: CSV, Excel (single and multi-sheet), JSON, clipboard  
4. Inspecting & understanding data  
5. Selecting, slicing, and extracting data  
6. Filtering and boolean indexing  
7. Adding, renaming, and dropping columns  
8. Sorting  
9. Handling missing values  
10. Duplicate management  
11. Data type conversion  
12. String operations  
13. Datetime operations  
14. GroupBy & aggregation  
15. Pivot tables & crosstabs  
16. Merge, join, and concatenate  
17. Apply, map, and custom functions  
18. Window functions — rolling & expanding  
19. Multi-level (hierarchical) indexing  
20. Reshaping — melt, stack, unstack, wide_to_long  
21. Performance — chunking, memory optimisation, categorical dtype  
22. Export: CSV, Excel, JSON, clipboard  
23. Capstone mini-projects

> Run cells from top to bottom. Every cell is designed to execute without error on a standard Python 3.8+ environment with pandas ≥ 1.5 and openpyxl installed.


---
## Section 1 — Environment Setup

In [133]:
# Install dependencies if running in a fresh environment
# !pip install pandas openpyxl xlrd matplotlib seaborn

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("pandas version :", pd.__version__)
print("numpy  version :", np.__version__)

# Display options — show more columns and rows in notebook output
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.width", 120)


pandas version : 2.3.1
numpy  version : 2.3.1


---
## Section 2 — The Series: Pandas' One-Dimensional Data Structure

In [134]:
# ── Creating a Series from a list ─────────────────────────────────────────────
s1 = pd.Series([10, 20, 30, 40, 50])
print("Default integer index:")
print(s1)
print()

# ── Custom index ─────────────────────────────────────────────────────────────
s2 = pd.Series(
    [250_000, 380_000, 175_000, 520_000, 310_000],
    index=["Lagos", "Abuja", "Kano", "Port Harcourt", "Ibadan"],
    name="Monthly Revenue (NGN)"
)
print("Series with custom string index:")
print(s2)


Default integer index:
0    10
1    20
2    30
3    40
4    50
dtype: int64

Series with custom string index:
Lagos            250000
Abuja            380000
Kano             175000
Port Harcourt    520000
Ibadan           310000
Name: Monthly Revenue (NGN), dtype: int64


In [135]:
# ── Creating a Series from a dictionary ──────────────────────────────────────
sales_dict = {"Q1": 1_200_000, "Q2": 980_000, "Q3": 1_450_000, "Q4": 1_600_000}
s_sales = pd.Series(sales_dict, name="Quarterly Sales")
print(s_sales)
print()
print("dtype  :", s_sales.dtype)
print("shape  :", s_sales.shape)
print("size   :", s_sales.size)
print("ndim   :", s_sales.ndim)
print("index  :", s_sales.index.tolist())
print("values :", s_sales.values)


Q1    1200000
Q2     980000
Q3    1450000
Q4    1600000
Name: Quarterly Sales, dtype: int64

dtype  : int64
shape  : (4,)
size   : 4
ndim   : 1
index  : ['Q1', 'Q2', 'Q3', 'Q4']
values : [1200000  980000 1450000 1600000]


In [136]:
# ── Arithmetic on a Series ───────────────────────────────────────────────────
print("Annual total       :", s_sales.sum())
print("Quarterly average  :", s_sales.mean())
print("Best quarter       :", s_sales.idxmax(), "→", s_sales.max())
print("Worst quarter      :", s_sales.idxmin(), "→", s_sales.min())
print()

# Scalar operations broadcast element-wise
print("Revenue + 10% bonus:")
print(s_sales * 1.10)


Annual total       : 5230000
Quarterly average  : 1307500.0
Best quarter       : Q4 → 1600000
Worst quarter      : Q2 → 980000

Revenue + 10% bonus:
Q1   1,320,000.00
Q2   1,078,000.00
Q3   1,595,000.00
Q4   1,760,000.00
Name: Quarterly Sales, dtype: float64


In [137]:
# ── Boolean masking on a Series ──────────────────────────────────────────────
above_avg = s_sales[s_sales > s_sales.mean()]
print("Quarters above average:")
print(above_avg)


Quarters above average:
Q3    1450000
Q4    1600000
Name: Quarterly Sales, dtype: int64


In [138]:
# ── Series with missing values ───────────────────────────────────────────────
s_missing = pd.Series([100, np.nan, 300, np.nan, 500], name="Sales with gaps")
print(s_missing)
print()
print("Missing count :", s_missing.isna().sum())
print("Filled forward:")
print(s_missing.ffill())
print()
print("Filled with mean:")
print(s_missing.fillna(s_missing.mean()))


0   100.00
1      NaN
2   300.00
3      NaN
4   500.00
Name: Sales with gaps, dtype: float64

Missing count : 2
Filled forward:
0   100.00
1   100.00
2   300.00
3   300.00
4   500.00
Name: Sales with gaps, dtype: float64

Filled with mean:
0   100.00
1   300.00
2   300.00
3   300.00
4   500.00
Name: Sales with gaps, dtype: float64


In [139]:
# ── Series string methods (.str accessor) ────────────────────────────────────
names = pd.Series(["adewale bello", "CHINWE OKAFOR", "  Emeka Nwosu  ", "fatima_yusuf"])
print("Original            :", names.tolist())
print("Title case          :", names.str.strip().str.title().tolist())
print("Contains 'a' (case) :", names.str.lower().str.contains("a").tolist())


Original            : ['adewale bello', 'CHINWE OKAFOR', '  Emeka Nwosu  ', 'fatima_yusuf']
Title case          : ['Adewale Bello', 'Chinwe Okafor', 'Emeka Nwosu', 'Fatima_Yusuf']
Contains 'a' (case) : [True, True, True, True]


---
## Section 3 — The DataFrame: Pandas' Two-Dimensional Data Structure

In [140]:
# ── Creating a DataFrame from a dictionary of lists ──────────────────────────
data = {
    "employee_id":  ["EMP001", "EMP002", "EMP003", "EMP004", "EMP005"],
    "name":         ["Adewale", "Chinwe", "Emeka", "Fatima", "Grace"],
    "department":   ["Finance", "IT", "Sales", "HR", "Operations"],
    "salary":       [450_000, 620_000, 390_000, 510_000, 480_000],
    "years_service":[3, 7, 1, 5, 4],
    "active":       [True, True, False, True, True],
}
df = pd.DataFrame(data)
df


,employee_id,name,department,salary,years_service,active
0,EMP001,Adewale,Finance,450000,3,True
1,EMP002,Chinwe,IT,620000,7,True
2,EMP003,Emeka,Sales,390000,1,False
3,EMP004,Fatima,HR,510000,5,True
4,EMP005,Grace,Operations,480000,4,True


In [141]:
# ── Core DataFrame attributes ─────────────────────────────────────────────────
print("Shape       :", df.shape)
print("Columns     :", df.columns.tolist())
print("Index       :", df.index.tolist())
print("Data types  :")
print(df.dtypes)
print()
print("Memory usage (bytes):", df.memory_usage(deep=True).sum())


Shape       : (5, 6)
Columns     : ['employee_id', 'name', 'department', 'salary', 'years_service', 'active']
Index       : [0, 1, 2, 3, 4]
Data types  :
employee_id      object
name             object
department       object
salary            int64
years_service     int64
active             bool
dtype: object

Memory usage (bytes): 1037


In [142]:
# ── Creating a DataFrame from a list of dictionaries ─────────────────────────
records = [
    {"city": "Lagos",         "population_M": 15.3, "gdp_usd_B": 136},
    {"city": "Kano",          "population_M": 3.8,  "gdp_usd_B": 12},
    {"city": "Ibadan",        "population_M": 3.6,  "gdp_usd_B": 9},
    {"city": "Abuja",         "population_M": 3.3,  "gdp_usd_B": 43},
    {"city": "Port Harcourt", "population_M": 1.9,  "gdp_usd_B": 18},
]
df_cities = pd.DataFrame(records)
df_cities


,city,population_M,gdp_usd_B
0,Lagos,15.30,136
1,Kano,3.80,12
2,Ibadan,3.60,9
3,Abuja,3.30,43
4,Port Harcourt,1.90,18


In [143]:
# ── Creating a DataFrame from NumPy arrays ────────────────────────────────────
np.random.seed(0)
df_np = pd.DataFrame(
    np.random.randint(50, 100, size=(5, 4)),
    columns=["Math", "English", "Science", "Social_Studies"],
    index=["Student_A", "Student_B", "Student_C", "Student_D", "Student_E"]
)
df_np


,Math,English,Science,Social_Studies
Student_A,94,97,50,53
Student_B,53,89,59,69
Student_C,71,86,73,56
Student_D,74,74,62,51
Student_E,88,89,73,96


---
## Section 4 — Reading Data into Pandas

### 4.1 — Reading a CSV file

In [144]:
# ── Basic CSV read ────────────────────────────────────────────────────────────
df_sales = pd.read_csv("sales_transactions.csv")
print(f"Shape: {df_sales.shape}")
df_sales.head()


Shape: (510, 13)


,transaction_id,date,region,category,sales_rep,quantity,unit_price,discount_pct,payment_method,customer_age,returned,satisfaction_score,revenue
0,TXN00001,2022-01-01 00:00:00.000000,West,Furniture,Fatima Yusuf,20,"20,021.08",0,Card,45.00,True,1.00,"400,421.60"
1,TXN00002,2022-01-03 04:39:55.190380,Central,Electronics,Fatima Yusuf,32,"27,696.00",20,Card,42.00,True,3.00,"709,017.60"
2,TXN00003,2022-01-05 09:19:50.380761,East,Sports,Grace Obi,36,"37,470.35",0,Cash,33.00,False,2.00,"1,348,932.60"
3,TXN00004,2022-01-07 13:59:45.571142,Central,Food & Beverage,James Ojo,4,"35,916.80",0,Card,19.00,False,2.00,"143,667.20"
4,TXN00005,2022-01-09 18:39:40.761523,Central,Food & Beverage,Grace Obi,27,"26,301.09",0,NaN,NaN,True,1.00,"710,129.43"


In [145]:
# ── read_csv with common parameters ──────────────────────────────────────────
df_sales = pd.read_csv(
    "sales_transactions.csv",
    parse_dates=["date"],          # parse date column automatically
    dtype={
        "transaction_id": str,
        "discount_pct":   float,
        "quantity":       int,
    },
    na_values=["", "N/A", "null", "NULL", "none"],  # treat these as NaN
)
print("Column dtypes after explicit typing:")
print(df_sales.dtypes)


Column dtypes after explicit typing:
transaction_id                object
date                  datetime64[ns]
region                        object
category                      object
sales_rep                     object
quantity                       int64
unit_price                   float64
discount_pct                 float64
payment_method                object
customer_age                 float64
returned                        bool
satisfaction_score           float64
revenue                      float64
dtype: object


In [146]:
# ── Reading only specific columns ─────────────────────────────────────────────
df_light = pd.read_csv(
    "sales_transactions.csv",
    usecols=["transaction_id", "date", "region", "category", "revenue"],
    parse_dates=["date"],
)
print("Columns loaded:", df_light.columns.tolist())
df_light.head(3)


Columns loaded: ['transaction_id', 'date', 'region', 'category', 'revenue']


,transaction_id,date,region,category,revenue
0,TXN00001,2022-01-01 00:00:00.000000,West,Furniture,"400,421.60"
1,TXN00002,2022-01-03 04:39:55.190380,Central,Electronics,"709,017.60"
2,TXN00003,2022-01-05 09:19:50.380761,East,Sports,"1,348,932.60"


In [147]:
# ── Reading with nrows — load only the first N rows ───────────────────────────
df_sample = pd.read_csv("sales_transactions.csv", nrows=20)
print("Rows loaded:", len(df_sample))


Rows loaded: 20


In [148]:
# ── Reading in chunks — useful for very large files ───────────────────────────
total_revenue = 0
chunk_count   = 0

for chunk in pd.read_csv("sales_transactions.csv", chunksize=100):
    # Convert to numeric safely before summing
    chunk["revenue"] = pd.to_numeric(chunk["revenue"], errors="coerce")
    total_revenue   += chunk["revenue"].sum()
    chunk_count     += 1

print(f"Processed {chunk_count} chunks")
print(f"Total revenue from chunked read: ₦{total_revenue:,.2f}")


Processed 6 chunks
Total revenue from chunked read: ₦297,741,372.66


### 4.2 — Reading Excel files (single and multiple sheets)

In [149]:
# ── Read the default (first) sheet ───────────────────────────────────────────
df_emp = pd.read_excel("pandas_training_data.xlsx")
print("Sheet loaded:", "Employees  (default first sheet)")
print("Shape:", df_emp.shape)
df_emp.head()


Sheet loaded: Employees  (default first sheet)
Shape: (200, 10)


,employee_id,full_name,department,job_title,hire_date,salary,age,gender,state,performance
0,EMP0001,Employee_1,Legal,Senior Manager,2015-01-01,2223521,52,Male,Lagos,Average
1,EMP0002,Employee_2,Sales,Manager,2015-01-18,797155,23,Male,Kano,Excellent
2,EMP0003,Employee_3,IT,Manager,2015-02-04,1488784,53,Male,Rivers,Good
3,EMP0004,Employee_4,HR,Manager,2015-02-21,683110,39,Female,Lagos,Excellent
4,EMP0005,Employee_5,Finance,Senior Manager,2015-03-11,479227,34,Female,Abuja,Good


In [150]:
# ── Read a specific sheet by name ─────────────────────────────────────────────
df_products = pd.read_excel("pandas_training_data.xlsx", sheet_name="Products")
print("Products sheet — shape:", df_products.shape)
df_products.head()


Products sheet — shape: (100, 10)


,product_id,product_name,category,subcategory,unit_cost,selling_price,stock_qty,reorder_level,supplier,active
0,PRD0001,Product_1,Food & Beverage,Standard,"6,172.91","11,280.15",181,19,SupplierD,False
1,PRD0002,Product_2,Furniture,Premium,"7,803.58","34,013.37",339,43,SupplierC,False
2,PRD0003,Product_3,Clothing,Standard,"17,117.11","40,259.00",64,35,SupplierD,True
3,PRD0004,Product_4,Clothing,Budget,"15,030.18","28,312.91",332,39,SupplierD,True
4,PRD0005,Product_5,Clothing,Standard,"5,591.84","37,173.72",0,18,SupplierB,True


In [151]:
# ── Read a specific sheet by index (0-based) ─────────────────────────────────
df_monthly = pd.read_excel("pandas_training_data.xlsx", sheet_name=2)
print("Sheet index 2 (Monthly_Sales) — shape:", df_monthly.shape)
df_monthly.head()


Sheet index 2 (Monthly_Sales) — shape: (36, 8)


,month,region,total_revenue,transactions,new_customers,returns,target_revenue,achievement_pct
0,Jan-2022,South,18932034,853,92,29,17352337,NaN
1,Feb-2022,South,19404956,701,165,19,23045551,NaN
2,Mar-2022,East,42725881,683,127,37,22177085,NaN
3,Apr-2022,South,34440349,937,88,30,37402527,NaN
4,May-2022,East,17394346,129,87,18,24883722,NaN


In [152]:
# ── Read ALL sheets at once into a dictionary ─────────────────────────────────
all_sheets = pd.read_excel("pandas_training_data.xlsx", sheet_name=None)

print("Available sheets:", list(all_sheets.keys()))
print()
for sheet_name, df_sheet in all_sheets.items():
    print(f"  {sheet_name:25s}  shape: {df_sheet.shape}")


Available sheets: ['Employees', 'Products', 'Monthly_Sales', 'Customer_Feedback', 'Budget_vs_Actuals']

  Employees                  shape: (200, 10)
  Products                   shape: (100, 10)
  Monthly_Sales              shape: (36, 8)
  Customer_Feedback          shape: (300, 9)
  Budget_vs_Actuals          shape: (21, 8)


In [153]:
# ── Access individual DataFrames from the dictionary ──────────────────────────
df_feedback = all_sheets["Customer_Feedback"]
df_budget   = all_sheets["Budget_vs_Actuals"]

print("Feedback sample:")
print(df_feedback.head(3))
print()
print("Budget vs Actuals sample:")
print(df_budget.head(3))


Feedback sample:
  feedback_id customer_id        date product_id  rating sentiment   channel  resolved  resolution_days
0     FB00001   CUST00655  2023-01-01    PRD0016       4  Positive     Phone      1.00            13.00
1     FB00002   CUST00115  2023-01-03    PRD0032       2  Positive  In-store      1.00            19.00
2     FB00003   CUST00026  2023-01-05    PRD0029       3  Positive  In-store       NaN              NaN

Budget vs Actuals sample:
   year  department  budget_ngn  actual_ngn  headcount_budget  headcount_actual  variance_ngn  variance_pct
0  2022     Finance     8273413     4094383                37                11           NaN           NaN
1  2022  Operations     2585750     3219995                34                16           NaN           NaN
2  2022          HR     6455950     3478863                16                24           NaN           NaN


In [154]:
# ── Read Excel with specific parameters ───────────────────────────────────────
df_emp_typed = pd.read_excel(
    "pandas_training_data.xlsx",
    sheet_name="Employees",
    dtype={
        "employee_id": str,
        "salary":      float,
        "age":         int,
    },
    na_values=["N/A", "", "null"],
)
print("Employees with explicit dtypes:")
print(df_emp_typed.dtypes)


Employees with explicit dtypes:
employee_id     object
full_name       object
department      object
job_title       object
hire_date       object
salary         float64
age              int64
gender          object
state           object
performance     object
dtype: object


### 4.3 — Reading JSON (inline demonstration)

In [155]:
import json, io

json_str = '''[
    {"product": "Laptop",  "price": 450000, "units_sold": 12},
    {"product": "Monitor", "price": 95000,  "units_sold": 30},
    {"product": "Keyboard","price": 18000,  "units_sold": 85}
]'''

df_json = pd.read_json(io.StringIO(json_str))
print(df_json)


    product   price  units_sold
0    Laptop  450000          12
1   Monitor   95000          30
2  Keyboard   18000          85


---
## Section 5 — Inspecting and Understanding Your Data

In [156]:
# Reload full CSV for demonstrations from here onward
df = pd.read_csv("sales_transactions.csv", parse_dates=["date"])

# ── First and last rows ───────────────────────────────────────────────────────
print("=== HEAD (first 5 rows) ===")
print(df.head())
print()
print("=== TAIL (last 5 rows) ===")
print(df.tail())


=== HEAD (first 5 rows) ===
  transaction_id                       date   region         category     sales_rep  quantity  unit_price  \
0       TXN00001 2022-01-01 00:00:00.000000     West        Furniture  Fatima Yusuf        20   20,021.08   
1       TXN00002 2022-01-03 04:39:55.190380  Central      Electronics  Fatima Yusuf        32   27,696.00   
2       TXN00003 2022-01-05 09:19:50.380761     East           Sports     Grace Obi        36   37,470.35   
3       TXN00004 2022-01-07 13:59:45.571142  Central  Food & Beverage     James Ojo         4   35,916.80   
4       TXN00005 2022-01-09 18:39:40.761523  Central  Food & Beverage     Grace Obi        27   26,301.09   

   discount_pct payment_method  customer_age  returned  satisfaction_score      revenue  
0             0           Card         45.00      True                1.00   400,421.60  
1            20           Card         42.00      True                3.00   709,017.60  
2             0           Cash         33.00   

In [157]:
# ── Random sample ────────────────────────────────────────────────────────────
print("Random sample of 5 rows:")
print(df.sample(5, random_state=1))


Random sample of 5 rows:
    transaction_id                       date   region         category      sales_rep  quantity  unit_price  \
47        TXN00048 2022-04-14 03:16:13.947895  Central         Clothing  Chinwe Okafor         2   32,506.99   
345       TXN00346 2024-01-28 01:32:20.681362    South           Sports     Ifeoma Eze         2   13,292.37   
284       TXN00285 2023-09-16 04:57:14.068136    North           Sports     Ifeoma Eze        45   22,442.02   
221       TXN00222 2023-04-30 23:02:17.074148    North  Food & Beverage    Emeka Nwosu        18   11,429.73   
501       TXN00341 2024-01-17 02:12:44.729458    North           Sports  Lanre Afolabi        15    1,129.39   

     discount_pct payment_method  customer_age  returned  satisfaction_score      revenue  
47              0           Card         29.00     False                5.00    65,013.98  
345             0           Cash         34.00     False                3.00    26,584.74  
284             0         

In [158]:
# ── DataFrame info ────────────────────────────────────────────────────────────
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 510 entries, 0 to 509
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   transaction_id      510 non-null    object        
 1   date                510 non-null    datetime64[ns]
 2   region              510 non-null    object        
 3   category            510 non-null    object        
 4   sales_rep           510 non-null    object        
 5   quantity            510 non-null    int64         
 6   unit_price          510 non-null    float64       
 7   discount_pct        510 non-null    int64         
 8   payment_method      416 non-null    object        
 9   customer_age        483 non-null    float64       
 10  returned            510 non-null    bool          
 11  satisfaction_score  469 non-null    float64       
 12  revenue             510 non-null    float64       
dtypes: bool(1), datetime64[ns](1), float64(4), int64(2

In [159]:
# ── Descriptive statistics ────────────────────────────────────────────────────
print("Numeric columns:")
print(df.describe())
print()
print("All columns including object types:")
print(df.describe(include="all"))


Numeric columns:
                                date  quantity  unit_price  discount_pct  customer_age  satisfaction_score  \
count                            510    510.00      510.00        510.00        483.00              469.00   
mean   2023-07-02 02:23:46.759400704     24.31   24,631.79          3.86         44.03                3.06   
min              2022-01-01 00:00:00      1.00      516.46          0.00         18.00                1.00   
25%    2022-09-28 11:00:07.214428160     12.00   11,904.40          0.00         31.00                2.00   
50%    2023-07-02 11:59:59.999999488     24.00   24,572.18          0.00         43.00                3.00   
75%    2024-03-31 03:40:02.404809216     36.00   36,969.70          5.00         57.00                4.00   
max              2024-12-31 00:00:00     49.00   49,776.28         20.00         69.00                5.00   
std                              NaN     14.11   14,092.43          6.66         15.01                1

In [160]:
# ── Value counts — frequency of each unique value ─────────────────────────────
print("Region distribution:")
print(df["region"].value_counts())
print()
print("Category distribution (normalised to %):")
print((df["category"].value_counts(normalize=True) * 100).round(1))


Region distribution:
region
West       113
North      111
South       96
East        95
Central     95
Name: count, dtype: int64

Category distribution (normalised to %):
category
Sports            22.20
Food & Beverage   20.00
Electronics       20.00
Clothing          19.20
Furniture         18.60
Name: proportion, dtype: float64


In [161]:
# ── Unique values and nunique ─────────────────────────────────────────────────
print("Unique regions:", df["region"].unique())
print("Unique categories:", df["category"].unique())
print()
print("Number of unique values per column:")
print(df.nunique())


Unique regions: ['West' 'Central' 'East' 'South' 'North']
Unique categories: ['Furniture' 'Electronics' 'Sports' 'Food & Beverage' 'Clothing']

Number of unique values per column:
transaction_id        500
date                  500
region                  5
category                5
sales_rep              10
quantity               49
unit_price            500
discount_pct            5
payment_method          4
customer_age           52
returned                2
satisfaction_score      5
revenue               500
dtype: int64


In [162]:
# ── Missing value audit ───────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report = missing_report[missing_report["missing_count"] > 0]
print("Columns with missing values:")
print(missing_report)


Columns with missing values:
                    missing_count  missing_pct
payment_method                 94        18.43
customer_age                   27         5.29
satisfaction_score             41         8.04


---
## Section 6 — Selecting, Slicing, and Extracting Data

### 6.1 — Selecting columns

In [163]:
# Single column → returns a Series
region_series = df["region"]
print(type(region_series))
print(region_series.head())
print()

# Single column as DataFrame (double brackets)
region_df = df[["region"]]
print(type(region_df))
print(region_df.head())


<class 'pandas.core.series.Series'>
0       West
1    Central
2       East
3    Central
4    Central
Name: region, dtype: object

<class 'pandas.core.frame.DataFrame'>
    region
0     West
1  Central
2     East
3  Central
4  Central


In [164]:
# Multiple columns → returns a DataFrame
subset = df[["transaction_id", "date", "region", "category", "revenue"]]
subset.head()


,transaction_id,date,region,category,revenue
0,TXN00001,2022-01-01 00:00:00.000000,West,Furniture,"400,421.60"
1,TXN00002,2022-01-03 04:39:55.190380,Central,Electronics,"709,017.60"
2,TXN00003,2022-01-05 09:19:50.380761,East,Sports,"1,348,932.60"
3,TXN00004,2022-01-07 13:59:45.571142,Central,Food & Beverage,"143,667.20"
4,TXN00005,2022-01-09 18:39:40.761523,Central,Food & Beverage,"710,129.43"


### 6.2 — .loc — label-based selection

In [165]:
# ── .loc[row_label, col_label] ────────────────────────────────────────────────

# Single row by index label
print("Row with index 0:")
print(df.loc[0])
print()

# Range of rows
print("Rows 0 to 3 (inclusive):")
print(df.loc[0:3, ["transaction_id","region","revenue"]])


Row with index 0:
transaction_id                   TXN00001
date                  2022-01-01 00:00:00
region                               West
category                        Furniture
sales_rep                    Fatima Yusuf
quantity                               20
unit_price                      20,021.08
discount_pct                            0
payment_method                       Card
customer_age                        45.00
returned                             True
satisfaction_score                   1.00
revenue                        400,421.60
Name: 0, dtype: object

Rows 0 to 3 (inclusive):
  transaction_id   region      revenue
0       TXN00001     West   400,421.60
1       TXN00002  Central   709,017.60
2       TXN00003     East 1,348,932.60
3       TXN00004  Central   143,667.20


In [166]:
# ── .loc with boolean condition ───────────────────────────────────────────────
high_value = df.loc[df["revenue"] > 500_000, ["transaction_id","date","region","revenue"]]
print(f"High-value transactions (>₦500,000): {len(high_value)} rows")
print(high_value.head())


High-value transactions (>₦500,000): 226 rows
  transaction_id                       date   region      revenue
1       TXN00002 2022-01-03 04:39:55.190380  Central   709,017.60
2       TXN00003 2022-01-05 09:19:50.380761     East 1,348,932.60
4       TXN00005 2022-01-09 18:39:40.761523  Central   710,129.43
5       TXN00006 2022-01-11 23:19:35.951903    South   700,543.99
7       TXN00008 2022-01-16 08:39:26.332665     East   839,393.63


In [167]:
# ── .loc — select all rows, specific columns ──────────────────────────────────
ids_and_rev = df.loc[:, ["transaction_id", "revenue"]]
ids_and_rev.head()


,transaction_id,revenue
0,TXN00001,"400,421.60"
1,TXN00002,"709,017.60"
2,TXN00003,"1,348,932.60"
3,TXN00004,"143,667.20"
4,TXN00005,"710,129.43"


### 6.3 — .iloc — position-based selection

In [168]:
# ── .iloc[row_position, col_position] ────────────────────────────────────────
print("First 3 rows, first 4 columns:")
print(df.iloc[0:3, 0:4])
print()

print("Last 5 rows, last 3 columns:")
print(df.iloc[-5:, -3:])


First 3 rows, first 4 columns:
  transaction_id                       date   region     category
0       TXN00001 2022-01-01 00:00:00.000000     West    Furniture
1       TXN00002 2022-01-03 04:39:55.190380  Central  Electronics
2       TXN00003 2022-01-05 09:19:50.380761     East       Sports

Last 5 rows, last 3 columns:
     returned  satisfaction_score      revenue
505     False                1.00    40,484.76
506     False                2.00   414,898.18
507     False                1.00   250,560.96
508     False                5.00   604,311.04
509     False                3.00 1,492,325.10


In [169]:
# ── Specific rows and columns by position ─────────────────────────────────────
print("Rows 10, 20, 50 — columns 0, 2, 6:")
print(df.iloc[[10, 20, 50], [0, 2, 6]])


Rows 10, 20, 50 — columns 0, 2, 6:
   transaction_id region  unit_price
10       TXN00011   West   32,907.31
20       TXN00021  South   26,846.88
50       TXN00051   West   45,341.61


### 6.4 — .at and .iat — single cell access

In [170]:
# .at — label-based single cell (fastest for one cell)
print("Cell at index 5, column 'region' :", df.at[5, "region"])

# .iat — position-based single cell
print("Cell at row 5, column 2           :", df.iat[5, 2])


Cell at index 5, column 'region' : South
Cell at row 5, column 2           : South


---
## Section 7 — Filtering and Boolean Indexing

In [171]:
# ── Simple single-condition filter ────────────────────────────────────────────
df_north = df[df["region"] == "North"]
print(f"North region transactions: {len(df_north)}")
df_north.head(3)


North region transactions: 111


,transaction_id,date,region,category,sales_rep,quantity,unit_price,discount_pct,payment_method,customer_age,returned,satisfaction_score,revenue
18,TXN00019,2022-02-09 11:58:33.426853,North,Sports,Grace Obi,36,"33,669.75",0,Card,47.00,False,4.00,"1,212,111.00"
23,TXN00024,2022-02-20 11:18:09.378757,North,Food & Beverage,Kemi Adeyemi,11,"36,022.97",0,NaN,36.00,False,2.00,"396,252.67"
24,TXN00025,2022-02-22 15:58:04.569138,North,Sports,Adewale Bello,33,"20,295.31",0,Transfer,69.00,False,4.00,"669,745.23"


In [172]:
# ── Multiple conditions with AND (&) ──────────────────────────────────────────
df_north_electronics = df[
    (df["region"] == "North") &
    (df["category"] == "Electronics")
]
print(f"North + Electronics: {len(df_north_electronics)} rows")
df_north_electronics.head(3)


North + Electronics: 22 rows


,transaction_id,date,region,category,sales_rep,quantity,unit_price,discount_pct,payment_method,customer_age,returned,satisfaction_score,revenue
87,TXN00088,2022-07-10 21:53:01.563126,North,Electronics,Chinwe Okafor,8,"45,323.48",20,NaN,38.00,False,2.00,"290,070.27"
91,TXN00092,2022-07-19 16:32:42.324649,North,Electronics,Chinwe Okafor,7,"29,989.66",0,Transfer,36.00,False,3.00,"209,927.62"
110,TXN00111,2022-08-30 09:11:10.941883,North,Electronics,Adewale Bello,48,"49,776.28",10,Transfer,30.00,False,5.00,"2,150,335.30"


In [173]:
# ── Multiple conditions with OR (|) ───────────────────────────────────────────
df_top_cats = df[
    (df["category"] == "Electronics") |
    (df["category"] == "Furniture")
]
print(f"Electronics OR Furniture: {len(df_top_cats)} rows")


Electronics OR Furniture: 197 rows


In [174]:
# ── Negation filter (~) ───────────────────────────────────────────────────────
df_not_south = df[~(df["region"] == "South")]
print(f"All regions except South: {len(df_not_south)} rows")


All regions except South: 414 rows


In [175]:
# ── .isin() — filter by a list of values ─────────────────────────────────────
target_regions = ["North", "East", "Central"]
df_selected = df[df["region"].isin(target_regions)]
print(f"Rows from North/East/Central: {len(df_selected)}")


Rows from North/East/Central: 301


In [176]:
# ── .between() — numeric range filter ────────────────────────────────────────
df_mid_rev = df[df["revenue"].between(100_000, 500_000)]
print(f"Revenue between ₦100K–₦500K: {len(df_mid_rev)} rows")
df_mid_rev[["transaction_id","revenue"]].head()


Revenue between ₦100K–₦500K: 193 rows


,transaction_id,revenue
0,TXN00001,"400,421.60"
3,TXN00004,"143,667.20"
6,TXN00007,"261,932.04"
9,TXN00010,"477,556.70"
11,TXN00012,"424,901.18"


In [177]:
# ── .query() — SQL-like string filtering ──────────────────────────────────────
df_q = df.query("region == 'North' and revenue > 200_000 and returned == False")
print(f"query() result: {len(df_q)} rows")
df_q.head(3)


query() result: 78 rows


,transaction_id,date,region,category,sales_rep,quantity,unit_price,discount_pct,payment_method,customer_age,returned,satisfaction_score,revenue
18,TXN00019,2022-02-09 11:58:33.426853,North,Sports,Grace Obi,36,"33,669.75",0,Card,47.00,False,4.00,"1,212,111.00"
23,TXN00024,2022-02-20 11:18:09.378757,North,Food & Beverage,Kemi Adeyemi,11,"36,022.97",0,NaN,36.00,False,2.00,"396,252.67"
24,TXN00025,2022-02-22 15:58:04.569138,North,Sports,Adewale Bello,33,"20,295.31",0,Transfer,69.00,False,4.00,"669,745.23"


In [178]:
# ── Filter on string content with .str.contains() ────────────────────────────
df_rep_a = df[df["sales_rep"].str.contains("Adewale", case=False, na=False)]
print(f"Transactions by Adewale: {len(df_rep_a)}")


Transactions by Adewale: 54


In [179]:
# ── Filter rows where a column is null / not null ────────────────────────────
df_missing_age  = df[df["customer_age"].isna()]
df_complete_age = df[df["customer_age"].notna()]
print(f"Missing age   : {len(df_missing_age)}")
print(f"Complete age  : {len(df_complete_age)}")


Missing age   : 27
Complete age  : 483


---
## Section 8 — Adding, Renaming, and Dropping Columns

In [180]:
# Work on a copy so originals stay clean
df_work = df.copy()

# ── Add a new calculated column ───────────────────────────────────────────────
df_work["discount_amount"] = (
    df_work["unit_price"] * df_work["quantity"] *
    df_work["discount_pct"] / 100
)
print("discount_amount column added:")
df_work[["transaction_id","unit_price","quantity","discount_pct","discount_amount"]].head(4)


discount_amount column added:


,transaction_id,unit_price,quantity,discount_pct,discount_amount
0,TXN00001,"20,021.08",20,0,0.00
1,TXN00002,"27,696.00",32,20,"177,254.40"
2,TXN00003,"37,470.35",36,0,0.00
3,TXN00004,"35,916.80",4,0,0.00


In [181]:
# ── Add a column derived from a condition (np.where) ─────────────────────────
df_work["revenue_band"] = np.where(
    df_work["revenue"] >= 500_000, "High",
    np.where(df_work["revenue"] >= 100_000, "Medium", "Low")
)
print(df_work["revenue_band"].value_counts())


revenue_band
High      226
Medium    193
Low        91
Name: count, dtype: int64


In [182]:
# ── Add a column with pd.cut — bin a numeric variable ────────────────────────
df_work["age_group"] = pd.cut(
    df_work["customer_age"],
    bins=[17, 25, 35, 45, 55, 70],
    labels=["18–25", "26–35", "36–45", "46–55", "56–70"],
    right=True
)
print("Age group distribution:")
print(df_work["age_group"].value_counts().sort_index())


Age group distribution:
age_group
18–25     67
26–35     95
36–45     96
46–55     85
56–70    140
Name: count, dtype: int64


In [183]:
# ── Rename columns ────────────────────────────────────────────────────────────
df_work = df_work.rename(columns={
    "unit_price":    "price_per_unit",
    "discount_pct":  "discount_percent",
    "returned":      "was_returned",
})
print("Renamed columns:", ["price_per_unit","discount_percent","was_returned"])


Renamed columns: ['price_per_unit', 'discount_percent', 'was_returned']


In [184]:
# ── Rename all columns at once (lowercasing) ──────────────────────────────────
df_lower = df_work.copy()
df_lower.columns = df_lower.columns.str.lower().str.replace(" ", "_")
print("Column names after lowercasing:", df_lower.columns.tolist())


Column names after lowercasing: ['transaction_id', 'date', 'region', 'category', 'sales_rep', 'quantity', 'price_per_unit', 'discount_percent', 'payment_method', 'customer_age', 'was_returned', 'satisfaction_score', 'revenue', 'discount_amount', 'revenue_band', 'age_group']


In [185]:
# ── Drop columns ──────────────────────────────────────────────────────────────
df_trimmed = df_work.drop(columns=["discount_amount", "age_group", "revenue_band"])
print("After dropping columns:", df_trimmed.columns.tolist())


After dropping columns: ['transaction_id', 'date', 'region', 'category', 'sales_rep', 'quantity', 'price_per_unit', 'discount_percent', 'payment_method', 'customer_age', 'was_returned', 'satisfaction_score', 'revenue']


In [186]:
# ── Drop rows ─────────────────────────────────────────────────────────────────
df_no_row0 = df_work.drop(index=[0, 1, 2])
print(f"Rows after dropping index 0,1,2: {len(df_no_row0)}")


Rows after dropping index 0,1,2: 507


In [187]:
# ── insert() — add a column at a specific position ────────────────────────────
df_ins = df.copy()
df_ins.insert(1, "source", "prodigy_training")
print("Columns after insert:", df_ins.columns[:4].tolist())


Columns after insert: ['transaction_id', 'source', 'date', 'region']


---
## Section 9 — Sorting

In [188]:
# ── Sort by a single column (ascending) ──────────────────────────────────────
df_sorted_rev = df.sort_values("revenue", ascending=False)
print("Top 5 by revenue:")
print(df_sorted_rev[["transaction_id","region","category","revenue"]].head())


Top 5 by revenue:
    transaction_id region         category      revenue
124       TXN00125  North         Clothing 2,306,346.40
110       TXN00111  North      Electronics 2,150,335.30
172       TXN00173  South           Sports 2,107,049.12
337       TXN00338   West         Clothing 2,043,875.68
307       TXN00308   West  Food & Beverage 2,009,053.09


In [189]:
# ── Sort by multiple columns ──────────────────────────────────────────────────
df_multi_sort = df.sort_values(
    ["region", "revenue"],
    ascending=[True, False]
)
print("Sorted by region (asc) then revenue (desc):")
df_multi_sort[["region","category","revenue"]].head(8)


Sorted by region (asc) then revenue (desc):


,region,category,revenue
478,Central,Food & Beverage,"1,977,150.13"
494,Central,Sports,"1,704,791.66"
401,Central,Electronics,"1,687,101.20"
444,Central,Food & Beverage,"1,646,874.06"
281,Central,Electronics,"1,634,724.38"
369,Central,Furniture,"1,447,179.82"
133,Central,Food & Beverage,"1,383,427.74"
296,Central,Sports,"1,359,459.36"


In [190]:
# ── Sort by index ─────────────────────────────────────────────────────────────
df_idx_sort = df.sort_index(ascending=False)
print("Last 3 rows after reverse-index sort:")
print(df_idx_sort.head(3).index.tolist())


Last 3 rows after reverse-index sort:
[509, 508, 507]


In [191]:
# ── nlargest / nsmallest ─────────────────────────────────────────────────────
print("Top 5 highest revenue transactions:")
print(df.nlargest(5, "revenue")[["transaction_id","date","category","revenue"]])
print()
print("5 smallest revenue transactions:")
print(df.nsmallest(5, "revenue")[["transaction_id","date","category","revenue"]])


Top 5 highest revenue transactions:
    transaction_id                       date         category      revenue
124       TXN00125 2022-09-30 02:30:03.607214         Clothing 2,306,346.40
110       TXN00111 2022-08-30 09:11:10.941883      Electronics 2,150,335.30
172       TXN00173 2023-01-13 10:26:12.745490           Sports 2,107,049.12
337       TXN00338 2024-01-10 12:12:59.158316         Clothing 2,043,875.68
307       TXN00308 2023-11-05 16:15:23.446893  Food & Beverage 2,009,053.09

5 smallest revenue transactions:
    transaction_id                       date     category  revenue
199       TXN00200 2023-03-13 16:24:02.885771     Clothing 3,713.98
291       TXN00292 2023-10-01 13:36:40.400801       Sports 4,521.52
477       TXN00478 2024-11-12 17:21:45.811623     Clothing 5,934.36
320       TXN00321 2023-12-04 04:54:20.921843  Electronics 6,699.06
42        TXN00043 2022-04-03 03:56:37.995991     Clothing 8,476.78


---
## Section 10 — Handling Missing Values

In [192]:
df_mv = df.copy()

# ── Detect missing values ─────────────────────────────────────────────────────
print("Total missing per column:")
print(df_mv.isnull().sum())
print()
print("Percentage missing per column:")
print((df_mv.isnull().mean() * 100).round(2))


Total missing per column:
transaction_id         0
date                   0
region                 0
category               0
sales_rep              0
quantity               0
unit_price             0
discount_pct           0
payment_method        94
customer_age          27
returned               0
satisfaction_score    41
revenue                0
dtype: int64

Percentage missing per column:
transaction_id        0.00
date                  0.00
region                0.00
category              0.00
sales_rep             0.00
quantity              0.00
unit_price            0.00
discount_pct          0.00
payment_method       18.43
customer_age          5.29
returned              0.00
satisfaction_score    8.04
revenue               0.00
dtype: float64


In [193]:
# ── Drop rows where ANY column has NaN ────────────────────────────────────────
df_dropped_any = df_mv.dropna()
print(f"Original rows     : {len(df_mv)}")
print(f"After dropna()    : {len(df_dropped_any)}")


Original rows     : 510
After dropna()    : 362


In [194]:
# ── Drop rows where ALL columns have NaN ─────────────────────────────────────
df_dropped_all = df_mv.dropna(how="all")
print(f"After dropna(how='all') : {len(df_dropped_all)}")


After dropna(how='all') : 510


In [195]:
# ── Drop rows only if specific columns have NaN ───────────────────────────────
df_dropped_key = df_mv.dropna(subset=["payment_method", "customer_age"])
print(f"After dropna on key cols: {len(df_dropped_key)}")


After dropna on key cols: 394


In [196]:
# ── fillna with a scalar value ────────────────────────────────────────────────
df_filled = df_mv.copy()
df_filled["payment_method"]   = df_filled["payment_method"].fillna("Unknown")
df_filled["satisfaction_score"] = df_filled["satisfaction_score"].fillna(
    df_filled["satisfaction_score"].median()
)
df_filled["customer_age"] = df_filled["customer_age"].fillna(
    df_filled["customer_age"].mean().round(0)
)
print("Missing after filling:")
print(df_filled[["payment_method","satisfaction_score","customer_age"]].isnull().sum())


Missing after filling:
payment_method        0
satisfaction_score    0
customer_age          0
dtype: int64


In [197]:
# ── Forward fill (ffill) and backward fill (bfill) ────────────────────────────
# These make most sense for time-series ordered data
df_ff = df_mv[["date","payment_method"]].sort_values("date").copy()
df_ff["payment_ffill"] = df_ff["payment_method"].ffill()
df_ff["payment_bfill"] = df_ff["payment_method"].bfill()
print("Forward and backward fill example:")
print(df_ff[df_ff["payment_method"].isna()].head(5))


Forward and backward fill example:
                         date payment_method payment_ffill payment_bfill
4  2022-01-09 18:39:40.761523            NaN          Card           POS
13 2022-01-29 12:38:57.474949            NaN          Card          Card
15 2022-02-02 21:58:47.855711            NaN          Card          Card
16 2022-02-05 02:38:43.046092            NaN          Card          Card
19 2022-02-11 16:38:28.617234            NaN          Card      Transfer


In [198]:
# ── interpolate — estimate missing numeric values ─────────────────────────────
s_interp = pd.Series([10.0, np.nan, np.nan, 40.0, np.nan, 60.0])
print("Original         :", s_interp.tolist())
print("Linear interp    :", s_interp.interpolate(method="linear").tolist())
print("Nearest interp   :", s_interp.interpolate(method="nearest").tolist())


Original         : [10.0, nan, nan, 40.0, nan, 60.0]
Linear interp    : [10.0, 20.0, 30.0, 40.0, 50.0, 60.0]
Nearest interp   : [10.0, 10.0, 40.0, 40.0, 40.0, 60.0]


---
## Section 11 — Duplicate Management

In [199]:
# ── Detect duplicates ─────────────────────────────────────────────────────────
dup_mask = df.duplicated()
print(f"Total duplicate rows (all columns): {dup_mask.sum()}")
print()

# Detect duplicates based on key columns only
dup_key = df.duplicated(subset=["transaction_id"])
print(f"Duplicate transaction IDs: {dup_key.sum()}")


Total duplicate rows (all columns): 10

Duplicate transaction IDs: 10


In [200]:
# ── View the duplicate rows ───────────────────────────────────────────────────
dupes = df[df.duplicated(keep=False)]  # keep=False marks ALL copies
print(f"Rows that are duplicates (both copies): {len(dupes)}")
dupes[["transaction_id","date","region","revenue"]].head(6)


Rows that are duplicates (both copies): 20


,transaction_id,date,region,revenue
31,TXN00032,2022-03-10 00:37:30.901803,West,"250,560.96"
47,TXN00048,2022-04-14 03:16:13.947895,Central,"65,013.98"
67,TXN00068,2022-05-28 00:34:37.755511,South,"27,999.24"
90,TXN00091,2022-07-17 11:52:47.134268,East,"1,492,325.10"
249,TXN00250,2023-07-01 09:40:02.404809,East,"604,311.04"
304,TXN00305,2023-10-30 02:15:37.875751,East,"774,097.20"


In [201]:
# ── Remove duplicates ─────────────────────────────────────────────────────────
df_dedup = df.drop_duplicates()
print(f"Original    : {len(df)}")
print(f"After dedup : {len(df_dedup)}")

# Keep the last occurrence instead of the first
df_dedup_last = df.drop_duplicates(keep="last")
print(f"After dedup (keep last): {len(df_dedup_last)}")


Original    : 510
After dedup : 500
After dedup (keep last): 500


In [202]:
# ── Remove duplicates on key columns only ────────────────────────────────────
df_dedup_key = df.drop_duplicates(subset=["transaction_id"], keep="first")
print(f"After dedup on transaction_id: {len(df_dedup_key)}")


After dedup on transaction_id: 500


---
## Section 12 — Data Type Conversion

In [203]:
df_types = df_dedup.copy()
print("Current dtypes:")
print(df_types.dtypes)


Current dtypes:
transaction_id                object
date                  datetime64[ns]
region                        object
category                      object
sales_rep                     object
quantity                       int64
unit_price                   float64
discount_pct                   int64
payment_method                object
customer_age                 float64
returned                        bool
satisfaction_score           float64
revenue                      float64
dtype: object


In [204]:
# ── astype() — explicit type conversion ──────────────────────────────────────
df_types["quantity"]    = df_types["quantity"].astype(int)
df_types["unit_price"]  = df_types["unit_price"].astype(float)
df_types["discount_pct"]= df_types["discount_pct"].astype(float)
df_types["returned"]    = df_types["returned"].astype(bool)

print("After astype conversions:")
print(df_types[["quantity","unit_price","discount_pct","returned"]].dtypes)


After astype conversions:
quantity          int64
unit_price      float64
discount_pct    float64
returned           bool
dtype: object


In [205]:
# ── pd.to_numeric with error coercion ────────────────────────────────────────
messy = pd.Series(["100", "200.5", "three", "400", None, "500"])
print("Original:", messy.tolist())
print("to_numeric (coerce):", pd.to_numeric(messy, errors="coerce").tolist())
print("to_numeric (ignore):", pd.to_numeric(messy, errors="ignore").tolist())


Original: ['100', '200.5', 'three', '400', None, '500']
to_numeric (coerce): [100.0, 200.5, nan, 400.0, nan, 500.0]
to_numeric (ignore): ['100', '200.5', 'three', '400', None, '500']


In [206]:
# ── pd.to_datetime ────────────────────────────────────────────────────────────
date_strings = pd.Series(["2024-01-15", "2024-02-29", "2024-12-01"])
dates_parsed = pd.to_datetime(date_strings, errors="coerce")
print("Parsed dates:", dates_parsed.tolist())
print("dtype:", dates_parsed.dtype)


Parsed dates: [Timestamp('2024-01-15 00:00:00'), Timestamp('2024-02-29 00:00:00'), Timestamp('2024-12-01 00:00:00')]
dtype: datetime64[ns]


In [207]:
# ── Categorical dtype — saves memory for low-cardinality string columns ────────
df_cat = df_types.copy()

mem_before = df_cat["region"].memory_usage(deep=True)
df_cat["region"]   = df_cat["region"].astype("category")
df_cat["category"] = df_cat["category"].astype("category")
mem_after  = df_cat["region"].memory_usage(deep=True)

print(f"region memory: {mem_before} bytes → {mem_after} bytes")
print(f"Reduction: {(1 - mem_after/mem_before)*100:.1f}%")
print("Categories:", df_cat["region"].cat.categories.tolist())


region memory: 30983 bytes → 4942 bytes
Reduction: 84.0%
Categories: ['Central', 'East', 'North', 'South', 'West']


---
## Section 13 — String Operations with .str Accessor

In [208]:
df_str = df_dedup.copy()

# ── Case manipulation ──────────────────────────────────────────────────────────
df_str["rep_upper"] = df_str["sales_rep"].str.upper()
df_str["rep_lower"] = df_str["sales_rep"].str.lower()
df_str["rep_title"] = df_str["sales_rep"].str.title()

df_str[["sales_rep","rep_upper","rep_lower","rep_title"]].head(3)


,sales_rep,rep_upper,rep_lower,rep_title
0,Fatima Yusuf,FATIMA YUSUF,fatima yusuf,Fatima Yusuf
1,Fatima Yusuf,FATIMA YUSUF,fatima yusuf,Fatima Yusuf
2,Grace Obi,GRACE OBI,grace obi,Grace Obi


In [209]:
# ── Split and extract parts ───────────────────────────────────────────────────
df_str["rep_first"] = df_str["sales_rep"].str.split().str[0]
df_str["rep_last"]  = df_str["sales_rep"].str.split().str[-1]
print(df_str[["sales_rep","rep_first","rep_last"]].head(4))


      sales_rep rep_first rep_last
0  Fatima Yusuf    Fatima    Yusuf
1  Fatima Yusuf    Fatima    Yusuf
2     Grace Obi     Grace      Obi
3     James Ojo     James      Ojo


In [210]:
# ── Contains, startswith, endswith ────────────────────────────────────────────
print("Reps containing 'Emeka':")
print(df_str[df_str["sales_rep"].str.contains("Emeka", na=False)]["sales_rep"].unique())

print("\nCategories starting with 'F':")
print(df_str[df_str["category"].str.startswith("F")]["category"].unique())


Reps containing 'Emeka':
['Emeka Nwosu']

Categories starting with 'F':
['Furniture' 'Food & Beverage']


In [211]:
# ── Replace and strip ─────────────────────────────────────────────────────────
messy_names = pd.Series(["  Lagos  ", "ABUJA", " Port Harcourt", "kano "])
cleaned = messy_names.str.strip().str.title()
print("Cleaned:", cleaned.tolist())
print()

# Replace text
df_str["region_abbrev"] = (
    df_str["region"]
    .str.replace("North", "N")
    .str.replace("South", "S")
    .str.replace("East",  "E")
    .str.replace("West",  "W")
    .str.replace("Central","C")
)
print(df_str["region_abbrev"].unique())


Cleaned: ['Lagos', 'Abuja', 'Port Harcourt', 'Kano']

['W' 'C' 'E' 'S' 'N']


In [212]:
# ── str.len() and str.count() ─────────────────────────────────────────────────
df_str["category_len"]  = df_str["category"].str.len()
df_str["spaces_in_rep"] = df_str["sales_rep"].str.count(" ")

print("Category character lengths:")
print(df_str.groupby("category")["category_len"].first())


Category character lengths:
category
Clothing            8
Electronics        11
Food & Beverage    15
Furniture           9
Sports              6
Name: category_len, dtype: int64


In [213]:
# ── str.extract with regex ─────────────────────────────────────────────────────
# Extract the numeric part from transaction_id e.g. "TXN00001" → "00001"
df_str["txn_number"] = df_str["transaction_id"].str.extract(r'TXN(\d+)')
print(df_str[["transaction_id","txn_number"]].head(4))


  transaction_id txn_number
0       TXN00001      00001
1       TXN00002      00002
2       TXN00003      00003
3       TXN00004      00004


---
## Section 14 — Datetime Operations with .dt Accessor

In [214]:
df_dt = df_dedup.copy()
df_dt["date"] = pd.to_datetime(df_dt["date"])

# ── Extract date components ────────────────────────────────────────────────────
df_dt["year"]        = df_dt["date"].dt.year
df_dt["month"]       = df_dt["date"].dt.month
df_dt["month_name"]  = df_dt["date"].dt.month_name()
df_dt["day"]         = df_dt["date"].dt.day
df_dt["day_of_week"] = df_dt["date"].dt.day_name()
df_dt["quarter"]     = df_dt["date"].dt.quarter
df_dt["week"]        = df_dt["date"].dt.isocalendar().week.astype(int)

df_dt[["date","year","month","month_name","quarter","day_of_week"]].head(4)


,date,year,month,month_name,quarter,day_of_week
0,2022-01-01 00:00:00.000000,2022,1,January,1,Saturday
1,2022-01-03 04:39:55.190380,2022,1,January,1,Monday
2,2022-01-05 09:19:50.380761,2022,1,January,1,Wednesday
3,2022-01-07 13:59:45.571142,2022,1,January,1,Friday


In [215]:
# ── Date arithmetic ───────────────────────────────────────────────────────────
df_dt["days_since"] = (pd.Timestamp("today") - df_dt["date"]).dt.days
print("Days since transaction (first 5 rows):")
print(df_dt[["date","days_since"]].head())


Days since transaction (first 5 rows):
                        date  days_since
0 2022-01-01 00:00:00.000000        1620
1 2022-01-03 04:39:55.190380        1618
2 2022-01-05 09:19:50.380761        1615
3 2022-01-07 13:59:45.571142        1613
4 2022-01-09 18:39:40.761523        1611


In [216]:
# ── Period-end dates ──────────────────────────────────────────────────────────
df_dt["month_end"]  = df_dt["date"] + pd.offsets.MonthEnd(0)
df_dt["quarter_end"]= df_dt["date"] + pd.offsets.QuarterEnd(0)
df_dt[["date","month_end","quarter_end"]].head(3)


,date,month_end,quarter_end
0,2022-01-01 00:00:00.000000,2022-01-31 00:00:00.000000,2022-03-31 00:00:00.000000
1,2022-01-03 04:39:55.190380,2022-01-31 04:39:55.190380,2022-03-31 04:39:55.190380
2,2022-01-05 09:19:50.380761,2022-01-31 09:19:50.380761,2022-03-31 09:19:50.380761


In [217]:
# ── Filtering by date range ───────────────────────────────────────────────────
start = pd.Timestamp("2023-01-01")
end   = pd.Timestamp("2023-12-31")
df_2023 = df_dt[(df_dt["date"] >= start) & (df_dt["date"] <= end)]
print(f"2023 transactions: {len(df_2023)}")

# Alternatively with .between
df_2023b = df_dt[df_dt["date"].between(start, end)]
print(f"2023 transactions (between): {len(df_2023b)}")


2023 transactions: 166
2023 transactions (between): 166


In [218]:
# ── Monthly revenue aggregation using date ────────────────────────────────────
df_dt["year_month"] = df_dt["date"].dt.to_period("M")
monthly_rev = df_dt.groupby("year_month")["revenue"].sum().reset_index()
monthly_rev.columns = ["period","total_revenue"]
print("Monthly revenue (first 12 months):")
print(monthly_rev.head(12))


Monthly revenue (first 12 months):
     period  total_revenue
0   2022-01   6,892,075.23
1   2022-02   7,734,540.08
2   2022-03   9,140,597.39
3   2022-04   6,272,613.18
4   2022-05   7,027,012.40
5   2022-06   7,415,105.69
6   2022-07   7,962,980.44
7   2022-08   8,551,999.47
8   2022-09   7,758,623.17
9   2022-10   9,543,189.13
10  2022-11   7,005,675.94
11  2022-12   8,193,629.90


---
## Section 15 — GroupBy and Aggregation

In [219]:
df_grp = df_dedup.copy()

# ── Single-column groupby with a single aggregation ───────────────────────────
rev_by_region = df_grp.groupby("region")["revenue"].sum().sort_values(ascending=False)
print("Total revenue by region:")
print(rev_by_region)


Total revenue by region:
region
North     67,913,648.47
West      64,663,567.37
South     59,146,362.38
Central   51,754,495.32
East      50,167,034.52
Name: revenue, dtype: float64


In [220]:
# ── Single-column groupby with multiple aggregations (.agg) ──────────────────
region_stats = (
    df_grp.groupby("region")["revenue"]
    .agg(
        total_revenue="sum",
        avg_revenue="mean",
        max_revenue="max",
        min_revenue="min",
        transaction_count="count",
    )
    .reset_index()
)
region_stats["avg_revenue"] = region_stats["avg_revenue"].round(0)
print(region_stats)


    region  total_revenue  avg_revenue  max_revenue  min_revenue  transaction_count
0  Central  51,754,495.32   556,500.00 1,977,150.13    10,101.96                 93
1     East  50,167,034.52   551,286.00 1,833,750.51    31,016.74                 91
2    North  67,913,648.47   623,061.00 2,306,346.40     4,521.52                109
3    South  59,146,362.38   622,593.00 2,107,049.12     6,699.06                 95
4     West  64,663,567.37   577,353.00 2,043,875.68     3,713.98                112


In [221]:
# ── Multi-column groupby ─────────────────────────────────────────────────────
region_cat = (
    df_grp.groupby(["region", "category"])
    .agg(
        transactions=("revenue", "count"),
        total_revenue=("revenue", "sum"),
        avg_qty=("quantity", "mean"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
print("Top 10 region-category combinations by revenue:")
print(region_cat.head(10))


Top 10 region-category combinations by revenue:
     region         category  transactions  total_revenue  avg_qty
10    North         Clothing            26  19,342,710.74    30.38
22     West  Food & Beverage            21  18,178,316.53    29.43
19    South           Sports            23  18,060,886.46    25.39
20     West         Clothing            29  16,073,767.00    24.76
4   Central           Sports            31  15,305,040.11    20.26
14    North           Sports            24  14,871,633.17    21.96
21     West      Electronics            26  14,786,518.78    27.58
11    North      Electronics            22  13,478,716.28    24.86
13    North        Furniture            19  12,399,272.79    30.32
17    South  Food & Beverage            18  12,364,776.03    28.28


In [222]:
# ── Named aggregation with different functions per column ─────────────────────
summary = df_grp.groupby("category").agg(
    num_transactions = ("transaction_id", "count"),
    total_revenue    = ("revenue",        "sum"),
    avg_unit_price   = ("unit_price",     "mean"),
    total_qty        = ("quantity",       "sum"),
    return_rate      = ("returned",       "mean"),
).reset_index()

summary["avg_unit_price"] = summary["avg_unit_price"].round(0)
summary["return_rate"]    = (summary["return_rate"] * 100).round(1)
print(summary)


          category  num_transactions  total_revenue  avg_unit_price  total_qty  return_rate
0         Clothing                95  56,262,349.52       24,322.00       2343         9.50
1      Electronics               101  59,299,070.03       24,967.00       2565        10.90
2  Food & Beverage                99  60,966,802.82       24,915.00       2483         5.10
3        Furniture                94  51,075,886.90       23,171.00       2318         8.50
4           Sports               111  66,040,998.79       25,634.00       2509        12.60


In [223]:
# ── groupby + transform — add group-level aggregate back to original row ────────
df_grp["region_total_rev"] = df_grp.groupby("region")["revenue"].transform("sum")
df_grp["region_share_pct"] = (df_grp["revenue"] / df_grp["region_total_rev"] * 100).round(3)
print("Revenue share within region (first 5 rows):")
print(df_grp[["transaction_id","region","revenue","region_total_rev","region_share_pct"]].head())


Revenue share within region (first 5 rows):
  transaction_id   region      revenue  region_total_rev  region_share_pct
0       TXN00001     West   400,421.60     64,663,567.37              0.62
1       TXN00002  Central   709,017.60     51,754,495.32              1.37
2       TXN00003     East 1,348,932.60     50,167,034.52              2.69
3       TXN00004  Central   143,667.20     51,754,495.32              0.28
4       TXN00005  Central   710,129.43     51,754,495.32              1.37


In [224]:
# ── groupby + filter — keep groups that meet a threshold ──────────────────────
big_regions = df_grp.groupby("region").filter(lambda g: g["revenue"].sum() > 500_000_000)
print(f"Rows in regions with total revenue >₦500M: {len(big_regions)}")
print("Regions kept:", big_regions["region"].unique())


Rows in regions with total revenue >₦500M: 0
Regions kept: []


In [225]:
# ── groupby size() vs count() ─────────────────────────────────────────────────
# size() counts ALL rows including NaN; count() ignores NaN per column
print("size()  — includes NaN rows:")
print(df_grp.groupby("region").size())
print()
print("count() — excludes NaN in each column:")
print(df_grp.groupby("region")["customer_age"].count())


size()  — includes NaN rows:
region
Central     93
East        91
North      109
South       95
West       112
dtype: int64

count() — excludes NaN in each column:
region
Central     87
East        85
North      102
South       92
West       107
Name: customer_age, dtype: int64


---
## Section 16 — Pivot Tables and Crosstabs

In [226]:
df_piv = df_dedup.copy()

# ── Basic pivot table ─────────────────────────────────────────────────────────
piv = pd.pivot_table(
    df_piv,
    values="revenue",
    index="region",
    columns="category",
    aggfunc="sum",
    fill_value=0,
    margins=True,        # adds row/column totals
    margins_name="TOTAL"
)
print("Revenue pivot — Region × Category:")
print(piv.round(0))


Revenue pivot — Region × Category:
category      Clothing   Electronics  Food & Beverage     Furniture        Sports          TOTAL
region                                                                                          
Central   6,663,170.00 10,996,040.00    11,132,860.00  7,657,385.00 15,305,040.00  51,754,495.00
East      6,449,189.00 10,292,082.00    11,469,535.00  9,839,291.00 12,116,938.00  50,167,035.00
North    19,342,711.00 13,478,716.00     7,821,315.00 12,399,273.00 14,871,633.00  67,913,648.00
South     7,733,512.00  9,745,713.00    12,364,776.00 11,241,475.00 18,060,886.00  59,146,362.00
West     16,073,767.00 14,786,519.00    18,178,317.00  9,938,464.00  5,686,502.00  64,663,567.00
TOTAL    56,262,350.00 59,299,070.00    60,966,803.00 51,075,887.00 66,040,999.00 293,645,108.00


In [227]:
# ── Pivot table with multiple aggregation functions ───────────────────────────
piv_multi = pd.pivot_table(
    df_piv,
    values="revenue",
    index="region",
    columns="category",
    aggfunc=["sum", "count"],
    fill_value=0,
)
piv_multi.columns = [f"{agg}_{cat}" for agg, cat in piv_multi.columns]
print("Multi-agg pivot (first two columns shown):")
print(piv_multi.iloc[:, :4])


Multi-agg pivot (first two columns shown):
         sum_Clothing  sum_Electronics  sum_Food & Beverage  sum_Furniture
region                                                                    
Central  6,663,170.25    10,996,039.78        11,132,859.80   7,657,385.38
East     6,449,189.35    10,292,082.02        11,469,534.97   9,839,290.63
North   19,342,710.74    13,478,716.28         7,821,315.49  12,399,272.79
South    7,733,512.18     9,745,713.17        12,364,776.03  11,241,474.54
West    16,073,767.00    14,786,518.78        18,178,316.53   9,938,463.56


In [228]:
# ── pd.crosstab — frequency table ────────────────────────────────────────────
ct = pd.crosstab(
    df_piv["region"],
    df_piv["category"],
    margins=True,
    margins_name="Total"
)
print("Transaction count — Region × Category:")
print(ct)


Transaction count — Region × Category:
category  Clothing  Electronics  Food & Beverage  Furniture  Sports  Total
region                                                                    
Central         12           18               18         14      31     93
East            14           16               24         18      19     91
North           26           22               18         19      24    109
South           14           19               18         21      23     95
West            29           26               21         22      14    112
Total           95          101               99         94     111    500


In [229]:
# ── Crosstab with percentages (normalised) ────────────────────────────────────
ct_pct = pd.crosstab(
    df_piv["region"],
    df_piv["returned"],
    normalize="index"      # % within each row (region)
).round(3) * 100

ct_pct.columns = ["Not Returned %", "Returned %"]
print("Return rate by region (%):")
print(ct_pct)


Return rate by region (%):
         Not Returned %  Returned %
region                             
Central           86.00       14.00
East              95.60        4.40
North             90.80        9.20
South             91.60        8.40
West              89.30       10.70


---
## Section 17 — Merge, Join, and Concatenate

In [230]:
# Create small sample DataFrames for merge demonstrations
orders = pd.DataFrame({
    "order_id":   ["O001","O002","O003","O004","O005"],
    "customer_id":["C001","C002","C003","C001","C004"],
    "amount":     [15000, 32000, 8500, 21000, 47000],
})

customers = pd.DataFrame({
    "customer_id": ["C001","C002","C003","C005"],
    "name":        ["Adewale", "Chinwe", "Emeka", "Fatima"],
    "city":        ["Lagos", "Abuja", "Kano", "Ibadan"],
})

print("Orders:"); print(orders)
print("\nCustomers:"); print(customers)


Orders:
  order_id customer_id  amount
0     O001        C001   15000
1     O002        C002   32000
2     O003        C003    8500
3     O004        C001   21000
4     O005        C004   47000

Customers:
  customer_id     name    city
0        C001  Adewale   Lagos
1        C002   Chinwe   Abuja
2        C003    Emeka    Kano
3        C005   Fatima  Ibadan


In [231]:
# ── INNER JOIN — only matching keys ──────────────────────────────────────────
inner = pd.merge(orders, customers, on="customer_id", how="inner")
print("INNER JOIN — matched rows only:")
print(inner)


INNER JOIN — matched rows only:
  order_id customer_id  amount     name   city
0     O001        C001   15000  Adewale  Lagos
1     O002        C002   32000   Chinwe  Abuja
2     O003        C003    8500    Emeka   Kano
3     O004        C001   21000  Adewale  Lagos


In [232]:
# ── LEFT JOIN — all orders, matched customer info where available ─────────────
left = pd.merge(orders, customers, on="customer_id", how="left")
print("LEFT JOIN — all orders kept:")
print(left)


LEFT JOIN — all orders kept:
  order_id customer_id  amount     name   city
0     O001        C001   15000  Adewale  Lagos
1     O002        C002   32000   Chinwe  Abuja
2     O003        C003    8500    Emeka   Kano
3     O004        C001   21000  Adewale  Lagos
4     O005        C004   47000      NaN    NaN


In [233]:
# ── RIGHT JOIN ────────────────────────────────────────────────────────────────
right = pd.merge(orders, customers, on="customer_id", how="right")
print("RIGHT JOIN — all customers kept:")
print(right)


RIGHT JOIN — all customers kept:
  order_id customer_id    amount     name    city
0     O001        C001 15,000.00  Adewale   Lagos
1     O004        C001 21,000.00  Adewale   Lagos
2     O002        C002 32,000.00   Chinwe   Abuja
3     O003        C003  8,500.00    Emeka    Kano
4      NaN        C005       NaN   Fatima  Ibadan


In [234]:
# ── OUTER (FULL) JOIN ─────────────────────────────────────────────────────────
outer = pd.merge(orders, customers, on="customer_id", how="outer")
print("OUTER JOIN — all rows from both:")
print(outer)


OUTER JOIN — all rows from both:
  order_id customer_id    amount     name    city
0     O001        C001 15,000.00  Adewale   Lagos
1     O004        C001 21,000.00  Adewale   Lagos
2     O002        C002 32,000.00   Chinwe   Abuja
3     O003        C003  8,500.00    Emeka    Kano
4     O005        C004 47,000.00      NaN     NaN
5      NaN        C005       NaN   Fatima  Ibadan


In [235]:
# ── Merge on different key names (left_on / right_on) ─────────────────────────
orders2    = orders.rename(columns={"customer_id": "cust_id"})
merged_diff = pd.merge(orders2, customers,
                       left_on="cust_id", right_on="customer_id", how="left")
print("Merge on different key names:")
print(merged_diff.head())


Merge on different key names:
  order_id cust_id  amount customer_id     name   city
0     O001    C001   15000        C001  Adewale  Lagos
1     O002    C002   32000        C002   Chinwe  Abuja
2     O003    C003    8500        C003    Emeka   Kano
3     O004    C001   21000        C001  Adewale  Lagos
4     O005    C004   47000         NaN      NaN    NaN


In [236]:
# ── Merge with suffixes when column names clash ───────────────────────────────
df_a = pd.DataFrame({"id": [1,2,3], "value": [10,20,30], "score": [1,2,3]})
df_b = pd.DataFrame({"id": [1,2,4], "value": [100,200,400], "score": [4,5,6]})
merged_sfx = pd.merge(df_a, df_b, on="id", suffixes=("_left","_right"))
print("Merge with suffixes:")
print(merged_sfx)


Merge with suffixes:
   id  value_left  score_left  value_right  score_right
0   1          10           1          100            4
1   2          20           2          200            5


In [237]:
# ── pd.concat — stacking DataFrames vertically ───────────────────────────────
df_q1 = pd.DataFrame({"month":["Jan","Feb","Mar"], "sales":[100,120,130]})
df_q2 = pd.DataFrame({"month":["Apr","May","Jun"], "sales":[140,115,160]})
df_q3 = pd.DataFrame({"month":["Jul","Aug","Sep"], "sales":[170,155,180]})

df_all = pd.concat([df_q1, df_q2, df_q3], ignore_index=True)
print("Vertical concat:")
print(df_all)


Vertical concat:
  month  sales
0   Jan    100
1   Feb    120
2   Mar    130
3   Apr    140
4   May    115
5   Jun    160
6   Jul    170
7   Aug    155
8   Sep    180


In [238]:
# ── pd.concat — horizontally (axis=1) ────────────────────────────────────────
df_names  = pd.DataFrame({"first": ["Ada","Bola","Chidi"]})
df_scores = pd.DataFrame({"math": [90, 75, 88], "english": [85, 92, 70]})
df_h = pd.concat([df_names, df_scores], axis=1)
print("Horizontal concat:")
print(df_h)


Horizontal concat:
   first  math  english
0    Ada    90       85
1   Bola    75       92
2  Chidi    88       70


In [239]:
# ── df.join — index-based join ────────────────────────────────────────────────
df_left  = pd.DataFrame({"A": [1,2,3]}, index=["x","y","z"])
df_right = pd.DataFrame({"B": [10,20,30]}, index=["x","y","w"])
print("df.join (left join on index by default):")
print(df_left.join(df_right))


df.join (left join on index by default):
   A     B
x  1 10.00
y  2 20.00
z  3   NaN


---
## Section 18 — apply(), map(), and Custom Functions

In [240]:
df_ap = df_dedup.copy()

# ── apply() on a single column ────────────────────────────────────────────────
def classify_revenue(val):
    if pd.isna(val):
        return "Unknown"
    elif val >= 1_000_000:
        return "Platinum"
    elif val >= 500_000:
        return "Gold"
    elif val >= 100_000:
        return "Silver"
    else:
        return "Bronze"

df_ap["rev_tier"] = df_ap["revenue"].apply(classify_revenue)
print("Revenue tier distribution:")
print(df_ap["rev_tier"].value_counts())


Revenue tier distribution:
rev_tier
Silver      190
Gold        117
Platinum    106
Bronze       87
Name: count, dtype: int64


In [241]:
# ── apply() with a lambda ─────────────────────────────────────────────────────
df_ap["qty_bucket"] = df_ap["quantity"].apply(lambda x: "Bulk" if x >= 30 else "Normal")
print(df_ap["qty_bucket"].value_counts())


qty_bucket
Normal    308
Bulk      192
Name: count, dtype: int64


In [242]:
# ── apply() across an entire row (axis=1) ────────────────────────────────────
def compute_net(row):
    gross = row["quantity"] * row["unit_price"]
    discount = gross * row["discount_pct"] / 100
    return round(gross - discount, 2)

df_ap["net_computed"] = df_ap.apply(compute_net, axis=1)
# Verify it matches the pre-computed revenue column
diff = (df_ap["net_computed"] - df_ap["revenue"]).abs().max()
print(f"Max difference from pre-computed revenue: {diff}")


Max difference from pre-computed revenue: 0.010000000009313226


In [243]:
# ── map() — element-wise replacement using a dict ────────────────────────────
region_map = {
    "North":   "Northern Zone",
    "South":   "Southern Zone",
    "East":    "Eastern Zone",
    "West":    "Western Zone",
    "Central": "Central Zone",
}
df_ap["region_full"] = df_ap["region"].map(region_map)
print(df_ap[["region","region_full"]].drop_duplicates())


     region    region_full
0      West   Western Zone
1   Central   Central Zone
2      East   Eastern Zone
5     South  Southern Zone
18    North  Northern Zone


In [244]:
# ── applymap (pandas <2.1) / map on DataFrame (pandas >=2.1) ─────────────────
# Apply a function element-wise to every cell of a numeric sub-DataFrame
df_nums = df_ap[["unit_price","revenue"]].head(5)

# Use map for DataFrame element-wise (replaces deprecated applymap)
try:
    result = df_nums.map(lambda x: round(x / 1000, 1) if pd.notna(x) else x)
except AttributeError:
    result = df_nums.applymap(lambda x: round(x / 1000, 1) if pd.notna(x) else x)

print("Values divided by 1000 (thousands):")
print(result)


Values divided by 1000 (thousands):
   unit_price  revenue
0       20.00   400.40
1       27.70   709.00
2       37.50 1,348.90
3       35.90   143.70
4       26.30   710.10


In [245]:
# ── np.vectorize — faster than apply for numeric transformations ──────────────
def discount_label(pct):
    if pct == 0:   return "No Discount"
    elif pct <= 10: return "Low"
    elif pct <= 15: return "Medium"
    else:           return "High"

vfunc = np.vectorize(discount_label)
df_ap["discount_label"] = vfunc(df_ap["discount_pct"].fillna(0))
print(df_ap["discount_label"].value_counts())


discount_label
No Discount    348
Low             71
High            42
Medium          39
Name: count, dtype: int64


---
## Section 19 — Window Functions: Rolling and Expanding

In [246]:
# Build a clean monthly time series first
df_ts = df_dedup.copy()
df_ts["date"] = pd.to_datetime(df_ts["date"])
df_ts["year_month"] = df_ts["date"].dt.to_period("M").dt.to_timestamp()
monthly = (
    df_ts.groupby("year_month")["revenue"]
    .sum()
    .reset_index()
    .sort_values("year_month")
    .rename(columns={"revenue": "monthly_revenue"})
)
print(f"Monthly series shape: {monthly.shape}")
monthly.head(8)


Monthly series shape: (36, 2)


,year_month,monthly_revenue
0,2022-01-01,"6,892,075.23"
1,2022-02-01,"7,734,540.08"
2,2022-03-01,"9,140,597.39"
3,2022-04-01,"6,272,613.18"
4,2022-05-01,"7,027,012.40"
5,2022-06-01,"7,415,105.69"
6,2022-07-01,"7,962,980.44"
7,2022-08-01,"8,551,999.47"


In [247]:
# ── Rolling mean (3-month moving average) ────────────────────────────────────
monthly["ma_3m"]  = monthly["monthly_revenue"].rolling(window=3).mean()
monthly["ma_6m"]  = monthly["monthly_revenue"].rolling(window=6).mean()
monthly["ma_12m"] = monthly["monthly_revenue"].rolling(window=12).mean()

print("Monthly revenue with moving averages:")
print(monthly[["year_month","monthly_revenue","ma_3m","ma_6m","ma_12m"]].tail(12))


Monthly revenue with moving averages:
   year_month  monthly_revenue         ma_3m        ma_6m       ma_12m
24 2024-01-01     9,177,175.28  7,156,041.16 8,336,078.20 8,073,209.20
25 2024-02-01     8,254,629.93  7,624,702.50 7,786,078.27 8,112,667.77
26 2024-03-01    10,147,315.48  9,193,040.23 8,099,476.28 8,502,329.81
27 2024-04-01     9,065,068.31  9,155,671.24 8,155,856.20 8,589,966.69
28 2024-05-01    12,224,942.92 10,479,108.90 9,051,905.70 8,882,254.31
29 2024-06-01     5,606,297.24  8,965,436.16 9,079,238.19 8,695,236.80
30 2024-07-01     6,791,562.68  8,207,600.95 8,681,636.09 8,508,857.15
31 2024-08-01     7,647,296.51  6,681,718.81 8,580,413.86 8,183,246.06
32 2024-09-01     7,645,249.17  7,361,369.45 8,163,402.80 8,131,439.54
33 2024-10-01    11,999,850.51  9,097,465.40 8,652,533.17 8,404,194.69
34 2024-11-01     6,734,004.52  8,793,034.73 7,737,376.77 8,394,641.24
35 2024-12-01     8,404,471.07  9,046,108.70 8,203,739.08 8,641,488.63


In [248]:
# ── Rolling sum — cumulative 3-month revenue window ───────────────────────────
monthly["rolling_3m_sum"] = monthly["monthly_revenue"].rolling(window=3).sum()
print("Rolling 3-month sum (last 6 months):")
print(monthly[["year_month","monthly_revenue","rolling_3m_sum"]].tail(6))


Rolling 3-month sum (last 6 months):
   year_month  monthly_revenue  rolling_3m_sum
30 2024-07-01     6,791,562.68   24,622,802.84
31 2024-08-01     7,647,296.51   20,045,156.43
32 2024-09-01     7,645,249.17   22,084,108.36
33 2024-10-01    11,999,850.51   27,292,396.19
34 2024-11-01     6,734,004.52   26,379,104.20
35 2024-12-01     8,404,471.07   27,138,326.10


In [249]:
# ── Expanding — cumulative running total ─────────────────────────────────────
monthly["cumulative_revenue"] = monthly["monthly_revenue"].expanding().sum()
monthly["running_avg"]        = monthly["monthly_revenue"].expanding().mean().round(0)
print("Cumulative revenue (last 6 months):")
print(monthly[["year_month","monthly_revenue","cumulative_revenue","running_avg"]].tail(6))


Cumulative revenue (last 6 months):
   year_month  monthly_revenue  cumulative_revenue  running_avg
30 2024-07-01     6,791,562.68      251,214,236.28 8,103,685.00
31 2024-08-01     7,647,296.51      258,861,532.79 8,089,423.00
32 2024-09-01     7,645,249.17      266,506,781.96 8,075,963.00
33 2024-10-01    11,999,850.51      278,506,632.47 8,191,372.00
34 2024-11-01     6,734,004.52      285,240,636.99 8,149,732.00
35 2024-12-01     8,404,471.07      293,645,108.06 8,156,809.00


In [250]:
# ── Rolling standard deviation — volatility measure ─────────────────────────
monthly["rev_volatility_3m"] = monthly["monthly_revenue"].rolling(3).std().round(0)
print("3-month revenue volatility (last 6 months):")
print(monthly[["year_month","monthly_revenue","rev_volatility_3m"]].tail(6))


3-month revenue volatility (last 6 months):
   year_month  monthly_revenue  rev_volatility_3m
30 2024-07-01     6,791,562.68       3,529,234.00
31 2024-08-01     7,647,296.51       1,024,924.00
32 2024-09-01     7,645,249.17         493,468.00
33 2024-10-01    11,999,850.51       2,513,539.00
34 2024-11-01     6,734,004.52       2,814,310.00
35 2024-12-01     8,404,471.07       2,690,921.00


In [251]:
# ── Shift — lag and lead values ───────────────────────────────────────────────
monthly["prev_month_rev"] = monthly["monthly_revenue"].shift(1)
monthly["next_month_rev"] = monthly["monthly_revenue"].shift(-1)
monthly["mom_growth_pct"] = (
    (monthly["monthly_revenue"] - monthly["prev_month_rev"]) /
    monthly["prev_month_rev"] * 100
).round(1)

print("Month-over-month growth (last 6 rows):")
print(monthly[["year_month","monthly_revenue","prev_month_rev","mom_growth_pct"]].tail(6))


Month-over-month growth (last 6 rows):
   year_month  monthly_revenue  prev_month_rev  mom_growth_pct
30 2024-07-01     6,791,562.68    5,606,297.24           21.10
31 2024-08-01     7,647,296.51    6,791,562.68           12.60
32 2024-09-01     7,645,249.17    7,647,296.51           -0.00
33 2024-10-01    11,999,850.51    7,645,249.17           57.00
34 2024-11-01     6,734,004.52   11,999,850.51          -43.90
35 2024-12-01     8,404,471.07    6,734,004.52           24.80


---
## Section 20 — Multi-level (Hierarchical) Indexing

In [252]:
# ── Create a MultiIndex DataFrame ────────────────────────────────────────────
df_mi = df_dedup.copy()
df_mi = (
    df_mi.groupby(["region","category"])
    .agg(
        total_rev=("revenue","sum"),
        count    =("revenue","count"),
        avg_rev  =("revenue","mean"),
    )
)
print("MultiIndex groupby result:")
print(df_mi.head(10))
print()
print("Index type:", type(df_mi.index))
print("Index names:", df_mi.index.names)


MultiIndex groupby result:
                            total_rev  count    avg_rev
region  category                                       
Central Clothing         6,663,170.25     12 555,264.19
        Electronics     10,996,039.78     18 610,891.10
        Food & Beverage 11,132,859.80     18 618,492.21
        Furniture        7,657,385.38     14 546,956.10
        Sports          15,305,040.11     31 493,710.97
East    Clothing         6,449,189.35     14 460,656.38
        Electronics     10,292,082.02     16 643,255.13
        Food & Beverage 11,469,534.97     24 477,897.29
        Furniture        9,839,290.63     18 546,627.26
        Sports          12,116,937.55     19 637,733.56

Index type: <class 'pandas.core.indexes.multi.MultiIndex'>
Index names: ['region', 'category']


In [253]:
# ── .loc on a MultiIndex ─────────────────────────────────────────────────────
print("North region — all categories:")
print(df_mi.loc["North"])
print()
print("North + Electronics only:")
print(df_mi.loc[("North", "Electronics")])


North region — all categories:
                    total_rev  count    avg_rev
category                                       
Clothing        19,342,710.74     26 743,950.41
Electronics     13,478,716.28     22 612,668.92
Food & Beverage  7,821,315.49     18 434,517.53
Furniture       12,399,272.79     19 652,593.30
Sports          14,871,633.17     24 619,651.38

North + Electronics only:
total_rev   13,478,716.28
count               22.00
avg_rev        612,668.92
Name: (North, Electronics), dtype: float64


In [254]:
# ── xs — cross-section selection ─────────────────────────────────────────────
print("Electronics across all regions:")
print(df_mi.xs("Electronics", level="category"))


Electronics across all regions:
            total_rev  count    avg_rev
region                                 
Central 10,996,039.78     18 610,891.10
East    10,292,082.02     16 643,255.13
North   13,478,716.28     22 612,668.92
South    9,745,713.17     19 512,932.27
West    14,786,518.78     26 568,712.26


In [255]:
# ── reset_index — flatten MultiIndex ─────────────────────────────────────────
df_flat = df_mi.reset_index()
print("Flattened (reset_index):")
print(df_flat.head(5))


Flattened (reset_index):
    region         category     total_rev  count    avg_rev
0  Central         Clothing  6,663,170.25     12 555,264.19
1  Central      Electronics 10,996,039.78     18 610,891.10
2  Central  Food & Beverage 11,132,859.80     18 618,492.21
3  Central        Furniture  7,657,385.38     14 546,956.10
4  Central           Sports 15,305,040.11     31 493,710.97


In [256]:
# ── set_index — promote columns to index ─────────────────────────────────────
df_single = df_dedup.copy()
df_indexed = df_single.set_index("transaction_id")
print("DataFrame with transaction_id as index:")
print(df_indexed.head(3))
print()
# reset back
df_reset = df_indexed.reset_index()
print("After reset_index:")
print(df_reset.head(2))


DataFrame with transaction_id as index:
                                     date   region     category     sales_rep  quantity  unit_price  discount_pct  \
transaction_id                                                                                                      
TXN00001       2022-01-01 00:00:00.000000     West    Furniture  Fatima Yusuf        20   20,021.08             0   
TXN00002       2022-01-03 04:39:55.190380  Central  Electronics  Fatima Yusuf        32   27,696.00            20   
TXN00003       2022-01-05 09:19:50.380761     East       Sports     Grace Obi        36   37,470.35             0   

               payment_method  customer_age  returned  satisfaction_score      revenue  
transaction_id                                                                          
TXN00001                 Card         45.00      True                1.00   400,421.60  
TXN00002                 Card         42.00      True                3.00   709,017.60  
TXN00003          

---
## Section 21 — Reshaping: melt, stack, unstack, wide_to_long

In [257]:
# ── Wide to long with melt ────────────────────────────────────────────────────
df_wide = pd.DataFrame({
    "employee":    ["Ada","Bola","Chidi"],
    "Q1_sales":    [120_000, 98_000, 145_000],
    "Q2_sales":    [135_000, 110_000, 128_000],
    "Q3_sales":    [150_000, 105_000, 162_000],
    "Q4_sales":    [175_000, 122_000, 191_000],
})
print("Wide format:")
print(df_wide)
print()

df_long = pd.melt(
    df_wide,
    id_vars="employee",
    value_vars=["Q1_sales","Q2_sales","Q3_sales","Q4_sales"],
    var_name="quarter",
    value_name="sales",
)
df_long["quarter"] = df_long["quarter"].str.replace("_sales","")
print("Long format after melt:")
print(df_long.sort_values(["employee","quarter"]))


Wide format:
  employee  Q1_sales  Q2_sales  Q3_sales  Q4_sales
0      Ada    120000    135000    150000    175000
1     Bola     98000    110000    105000    122000
2    Chidi    145000    128000    162000    191000

Long format after melt:
   employee quarter   sales
0       Ada      Q1  120000
3       Ada      Q2  135000
6       Ada      Q3  150000
9       Ada      Q4  175000
1      Bola      Q1   98000
4      Bola      Q2  110000
7      Bola      Q3  105000
10     Bola      Q4  122000
2     Chidi      Q1  145000
5     Chidi      Q2  128000
8     Chidi      Q3  162000
11    Chidi      Q4  191000


In [258]:
# ── pivot — long to wide ─────────────────────────────────────────────────────
df_back_wide = df_long.pivot(
    index="employee",
    columns="quarter",
    values="sales"
)
df_back_wide.columns.name = None
df_back_wide = df_back_wide.reset_index()
print("Back to wide via pivot:")
print(df_back_wide)


Back to wide via pivot:
  employee      Q1      Q2      Q3      Q4
0      Ada  120000  135000  150000  175000
1     Bola   98000  110000  105000  122000
2    Chidi  145000  128000  162000  191000


In [259]:
# ── stack and unstack ─────────────────────────────────────────────────────────
scores = pd.DataFrame(
    [[85, 90, 78], [92, 88, 95], [74, 83, 80]],
    index=["Student_A","Student_B","Student_C"],
    columns=["Math","English","Science"]
)
print("Original wide DataFrame:")
print(scores)
print()

stacked = scores.stack()
print("After stack():")
print(stacked)
print()

unstacked = stacked.unstack()
print("After unstack() — back to wide:")
print(unstacked)


Original wide DataFrame:
           Math  English  Science
Student_A    85       90       78
Student_B    92       88       95
Student_C    74       83       80

After stack():
Student_A  Math       85
           English    90
           Science    78
Student_B  Math       92
           English    88
           Science    95
Student_C  Math       74
           English    83
           Science    80
dtype: int64

After unstack() — back to wide:
           Math  English  Science
Student_A    85       90       78
Student_B    92       88       95
Student_C    74       83       80


In [260]:
# ── pd.wide_to_long ───────────────────────────────────────────────────────────
df_rev = pd.DataFrame({
    "dept":        ["Finance", "IT", "HR"],
    "rev_2022":    [1_200_000, 980_000, 640_000],
    "rev_2023":    [1_350_000, 1_100_000, 720_000],
    "rev_2024":    [1_500_000, 1_250_000, 810_000],
    "cost_2022":   [900_000, 720_000, 500_000],
    "cost_2023":   [950_000, 780_000, 530_000],
    "cost_2024":   [1_000_000, 850_000, 570_000],
})
df_rev["id"] = range(len(df_rev))

df_long2 = pd.wide_to_long(
    df_rev,
    stubnames=["rev", "cost"],
    i="id",
    j="year",
    sep="_",
    suffix=r"\d+"
).reset_index().drop("id", axis=1)

print(df_long2.sort_values(["dept", "year"]).reset_index(drop=True))

   year     dept      rev     cost
0  2022  Finance  1200000   900000
1  2023  Finance  1350000   950000
2  2024  Finance  1500000  1000000
3  2022       HR   640000   500000
4  2023       HR   720000   530000
5  2024       HR   810000   570000
6  2022       IT   980000   720000
7  2023       IT  1100000   780000
8  2024       IT  1250000   850000


---
## Section 22 — Performance: Memory Optimisation and Efficient Patterns

In [261]:
# ── Memory usage overview ─────────────────────────────────────────────────────
df_perf = df_dedup.copy()
print("Memory usage by column (deep=True):")
mem = df_perf.memory_usage(deep=True).sort_values(ascending=False)
print(mem)
print(f"\nTotal: {mem.sum():,} bytes ({mem.sum()/1024:.1f} KB)")


Memory usage by column (deep=True):
sales_rep             30136
category              29368
transaction_id        28500
region                26983
payment_method        24901
Index                  4000
date                   4000
quantity               4000
unit_price             4000
discount_pct           4000
customer_age           4000
satisfaction_score     4000
revenue                4000
returned                500
dtype: int64

Total: 172,388 bytes (168.3 KB)


In [262]:
# ── Reduce memory with appropriate dtypes ────────────────────────────────────
df_opt = df_dedup.copy()

# Downcast integers
int_cols = df_opt.select_dtypes("int64").columns
df_opt[int_cols] = df_opt[int_cols].apply(pd.to_numeric, downcast="integer")

# Downcast floats
float_cols = df_opt.select_dtypes("float64").columns
df_opt[float_cols] = df_opt[float_cols].apply(pd.to_numeric, downcast="float")

# Convert low-cardinality strings to category
for col in ["region","category","payment_method"]:
    df_opt[col] = df_opt[col].astype("category")

mem_before = df_dedup.memory_usage(deep=True).sum()
mem_after  = df_opt.memory_usage(deep=True).sum()
print(f"Memory before optimisation: {mem_before:,} bytes")
print(f"Memory after  optimisation: {mem_after:,} bytes")
print(f"Reduction: {(1 - mem_after/mem_before)*100:.1f}%")


Memory before optimisation: 172,388 bytes
Memory after  optimisation: 82,931 bytes
Reduction: 51.9%


In [263]:
# ── Vectorised operations vs apply — speed comparison ─────────────────────────
import time

df_speed = df_dedup.copy()

# Method 1: apply (slow for numeric work)
t0 = time.time()
r1 = df_speed["revenue"].apply(lambda x: x * 1.075)
t1 = time.time()

# Method 2: vectorised (fast)
t2 = time.time()
r2 = df_speed["revenue"] * 1.075
t3 = time.time()

print(f"apply()     : {(t1-t0)*1000:.2f} ms")
print(f"vectorised  : {(t3-t2)*1000:.2f} ms")
print(f"Speedup     : {(t1-t0)/(t3-t2):.0f}x")


apply()     : 0.90 ms
vectorised  : 0.39 ms
Speedup     : 2x


In [264]:
# ── Use .query() instead of chained boolean filters for readability ───────────
%timeit df_dedup[(df_dedup["region"]=="North") & (df_dedup["revenue"]>200000)]
%timeit df_dedup.query("region == 'North' and revenue > 200000")


695 μs ± 91.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
4.05 ms ± 843 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


---
## Section 23 — Exporting Data

In [265]:
df_export = df_dedup.head(50).copy()

# ── Export to CSV ─────────────────────────────────────────────────────────────
df_export.to_csv("output_sample.csv", index=False)
print("Saved: output_sample.csv")

# With specific encoding (important for Nigerian character data)
df_export.to_csv("output_utf8.csv", index=False, encoding="utf-8")

# Append to an existing CSV (useful for streaming data)
df_export.iloc[:5].to_csv("output_append.csv", index=False, mode="w", header=True)
df_export.iloc[5:10].to_csv("output_append.csv", index=False, mode="a", header=False)
print("Append test done.")


Saved: output_sample.csv
Append test done.


In [266]:
# ── Export to Excel — single sheet ────────────────────────────────────────────
df_export.to_excel("output_single.xlsx", index=False, sheet_name="Sales_Sample")
print("Saved: output_single.xlsx")


Saved: output_single.xlsx


In [267]:
# ── Export to Excel — multiple sheets with ExcelWriter ────────────────────────
region_summary = df_dedup.groupby("region")["revenue"].agg(["sum","mean","count"]).reset_index()
cat_summary    = df_dedup.groupby("category")["revenue"].agg(["sum","mean","count"]).reset_index()

with pd.ExcelWriter("output_multi_sheet.xlsx", engine="openpyxl") as writer:
    df_export.to_excel(writer, sheet_name="Raw_Transactions",  index=False)
    region_summary.to_excel(writer, sheet_name="Region_Summary",   index=False)
    cat_summary.to_excel(   writer, sheet_name="Category_Summary", index=False)

print("Saved: output_multi_sheet.xlsx with 3 sheets")


Saved: output_multi_sheet.xlsx with 3 sheets


In [268]:
# ── Export to JSON ────────────────────────────────────────────────────────────
df_export.head(10).to_json("output.json", orient="records", indent=2, date_format="iso")
print("Saved: output.json")

# Verify by reading back
import json
with open("output.json") as f:
    records = json.load(f)
print(f"Records in JSON: {len(records)}")
print("First record keys:", list(records[0].keys()))


Saved: output.json
Records in JSON: 10
First record keys: ['transaction_id', 'date', 'region', 'category', 'sales_rep', 'quantity', 'unit_price', 'discount_pct', 'payment_method', 'customer_age', 'returned', 'satisfaction_score', 'revenue']


---
## Section 24 — Capstone Mini-Projects

### Project 1: Full Sales Pipeline

In [269]:
# ── CAPSTONE 1: End-to-end sales analysis pipeline ────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# STEP 1: Load
df_cap = pd.read_csv("sales_transactions.csv", parse_dates=["date"])
print(f"Raw data: {df_cap.shape}")

# STEP 2: Deduplicate
df_cap = df_cap.drop_duplicates(subset=["transaction_id"], keep="first")
print(f"After dedup: {df_cap.shape}")

# STEP 3: Handle missing values
df_cap["payment_method"]    = df_cap["payment_method"].fillna("Unknown")
df_cap["customer_age"]      = df_cap["customer_age"].fillna(df_cap["customer_age"].median())
df_cap["satisfaction_score"]= df_cap["satisfaction_score"].fillna(df_cap["satisfaction_score"].median())

# STEP 4: Engineer features
df_cap["year"]        = df_cap["date"].dt.year
df_cap["month"]       = df_cap["date"].dt.month
df_cap["quarter"]     = "Q" + df_cap["date"].dt.quarter.astype(str)
df_cap["year_month"]  = df_cap["date"].dt.to_period("M").dt.to_timestamp()
df_cap["revenue_band"]= pd.cut(df_cap["revenue"],
                                bins=[0,100_000,500_000,1_000_000,np.inf],
                                labels=["Low","Medium","High","Platinum"])

# STEP 5: Summary tables
print("\n=== ANNUAL REVENUE BY REGION ===")
annual_region = (
    df_cap.groupby(["year","region"])["revenue"]
    .sum().unstack("region").fillna(0)
    .astype(int)
)
print(annual_region)

print("\n=== CATEGORY PERFORMANCE ===")
cat_perf = df_cap.groupby("category").agg(
    txn_count       = ("transaction_id","count"),
    total_revenue   = ("revenue","sum"),
    avg_revenue     = ("revenue","mean"),
    return_rate_pct = ("returned","mean"),
).assign(
    avg_revenue     = lambda d: d["avg_revenue"].round(0),
    return_rate_pct = lambda d: (d["return_rate_pct"] * 100).round(1),
    total_revenue   = lambda d: d["total_revenue"].astype(int),
).sort_values("total_revenue", ascending=False)
print(cat_perf)


Raw data: (510, 13)
After dedup: (500, 13)

=== ANNUAL REVENUE BY REGION ===
region   Central      East     North     South      West
year                                                    
2022    14643675  16258433  23401141  17046866  22147925
2023    11920955  20548093  25131484  17200767  21647900
2024    25189864  13360507  19381022  24898728  20867740

=== CATEGORY PERFORMANCE ===
                 txn_count  total_revenue  avg_revenue  return_rate_pct
category                                                               
Sports                 111       66040998   594,964.00            12.60
Food & Beverage         99       60966802   615,826.00             5.10
Electronics            101       59299070   587,120.00            10.90
Clothing                95       56262349   592,235.00             9.50
Furniture               94       51075886   543,360.00             8.50


In [270]:
print("=== TOP 5 SALES REPS ===")
top_reps = (
    df_cap.groupby("sales_rep").agg(
        total_revenue = ("revenue","sum"),
        transactions  = ("transaction_id","count"),
        avg_score     = ("satisfaction_score","mean"),
    )
    .assign(
        total_revenue = lambda d: d["total_revenue"].astype(int),
        avg_score     = lambda d: d["avg_score"].round(2),
    )
    .sort_values("total_revenue", ascending=False)
    .head(5)
)
print(top_reps)

print("\n=== MONTHLY TREND (2023) ===")
monthly_2023 = (
    df_cap[df_cap["year"] == 2023]
    .groupby("year_month")["revenue"]
    .sum()
    .reset_index()
    .rename(columns={"revenue":"monthly_revenue"})
)
monthly_2023["mom_growth"] = monthly_2023["monthly_revenue"].pct_change().mul(100).round(1)
print(monthly_2023)


=== TOP 5 SALES REPS ===
               total_revenue  transactions  avg_score
sales_rep                                            
James Ojo           37763823            54       2.83
Lanre Afolabi       35749170            50       2.92
Grace Obi           34435720            53       2.85
Adewale Bello       33630221            53       3.00
Ifeoma Eze          31351375            53       3.04

=== MONTHLY TREND (2023) ===
   year_month  monthly_revenue  mom_growth
0  2023-01-01     8,747,867.29         NaN
1  2023-02-01     7,781,127.07      -11.10
2  2023-03-01     5,471,371.06      -29.70
3  2023-04-01     8,013,425.69       46.50
4  2023-05-01     8,717,491.45        8.80
5  2023-06-01     7,850,507.45       -9.90
6  2023-07-01     9,028,118.48       15.00
7  2023-08-01    11,554,629.52       28.00
8  2023-09-01     8,266,927.41      -28.50
9  2023-10-01     8,726,788.81        5.60
10 2023-11-01     6,848,645.91      -21.50
11 2023-12-01     5,442,302.28      -20.50


### Project 2: Multi-sheet Excel Analysis

In [271]:
# ── CAPSTONE 2: Analyse the multi-sheet Excel file ───────────────────────────
sheets = pd.read_excel("pandas_training_data.xlsx", sheet_name=None)

df_e  = sheets["Employees"]
df_p  = sheets["Products"]
df_ms = sheets["Monthly_Sales"]
df_fb = sheets["Customer_Feedback"]
df_bv = sheets["Budget_vs_Actuals"]

# ── HR Analysis: salary bands by department ───────────────────────────────────
df_e["salary"] = pd.to_numeric(df_e["salary"], errors="coerce")
hr_summary = (
    df_e.groupby("department")["salary"]
    .agg(avg_salary="mean", min_salary="min", max_salary="max", headcount="count")
    .assign(
        avg_salary = lambda d: d["avg_salary"].round(0),
        min_salary = lambda d: d["min_salary"].astype(int),
        max_salary = lambda d: d["max_salary"].astype(int),
    )
    .sort_values("avg_salary", ascending=False)
)
print("=== SALARY ANALYSIS BY DEPARTMENT ===")
print(hr_summary)


=== SALARY ANALYSIS BY DEPARTMENT ===
             avg_salary  min_salary  max_salary  headcount
department                                                
Legal      1,618,134.00      455545     2428736         31
Operations 1,446,873.00      288054     2481155         29
HR         1,431,874.00      376481     2498538         28
Finance    1,413,056.00      337764     2449236         24
IT         1,272,015.00      289371     2164419         35
Sales      1,201,119.00      313273     2467602         31
Marketing  1,199,153.00      325203     2300021         22


In [272]:
# ── Product Analysis: margin by category ──────────────────────────────────────
df_p["unit_cost"]     = pd.to_numeric(df_p["unit_cost"],     errors="coerce")
df_p["selling_price"] = pd.to_numeric(df_p["selling_price"], errors="coerce")
df_p["gross_margin"]  = (df_p["selling_price"] - df_p["unit_cost"]) / df_p["selling_price"] * 100

prod_summary = (
    df_p.groupby("category").agg(
        products       = ("product_id","count"),
        avg_margin_pct = ("gross_margin","mean"),
        total_stock    = ("stock_qty","sum"),
        below_reorder  = ("stock_qty", lambda x: (x <= df_p.loc[x.index,"reorder_level"]).sum()),
    )
    .assign(avg_margin_pct = lambda d: d["avg_margin_pct"].round(1))
    .sort_values("avg_margin_pct", ascending=False)
)
print("=== PRODUCT MARGIN BY CATEGORY ===")
print(prod_summary)


=== PRODUCT MARGIN BY CATEGORY ===
                 products  avg_margin_pct  total_stock  below_reorder
category                                                             
Electronics            19           53.10         4313              1
Furniture              21           32.30         4974              2
Sports                 20           17.80         5332              2
Clothing               18           10.00         3300              4
Food & Beverage        22          -34.10         6414              1


In [273]:
# ── Budget vs Actuals: variance analysis ─────────────────────────────────────
df_bv["budget_ngn"] = pd.to_numeric(df_bv["budget_ngn"], errors="coerce")
df_bv["actual_ngn"] = pd.to_numeric(df_bv["actual_ngn"], errors="coerce")
df_bv["variance"]   = df_bv["actual_ngn"] - df_bv["budget_ngn"]
df_bv["variance_pct"] = (df_bv["variance"] / df_bv["budget_ngn"] * 100).round(1)

over_budget = df_bv[df_bv["variance"] > 0].sort_values("variance_pct", ascending=False)
print("=== OVER-BUDGET LINES (Top 10) ===")
print(over_budget[["year","department","budget_ngn","actual_ngn","variance","variance_pct"]].head(10))


=== OVER-BUDGET LINES (Top 10) ===
    year  department  budget_ngn  actual_ngn  variance  variance_pct
12  2023       Sales     1108629     9869017   8760388        790.20
8   2023  Operations     2692527     9020895   6328368        235.00
7   2023     Finance     3006550     8180523   5173973        172.10
13  2023       Legal     5670436     9421301   3750865         66.10
16  2024          HR     6305237     9810167   3504930         55.60
5   2022       Sales     3840883     5932836   2091953         54.50
11  2023   Marketing     3374255     5151237   1776982         52.70
3   2022          IT     4032297     5945095   1912798         47.40
4   2022   Marketing     6269373     8389304   2119931         33.80
1   2022  Operations     2585750     3219995    634245         24.50


In [274]:
# ── Feedback Analysis: sentiment and resolution ───────────────────────────────
df_fb["rating"] = pd.to_numeric(df_fb["rating"], errors="coerce")
fb_summary = (
    df_fb.groupby("sentiment").agg(
        count           = ("feedback_id","count"),
        avg_rating      = ("rating","mean"),
        resolved_rate   = ("resolved", lambda x: pd.to_numeric(x, errors="coerce").mean()),
        avg_res_days    = ("resolution_days","mean"),
    )
    .assign(
        avg_rating    = lambda d: d["avg_rating"].round(2),
        resolved_rate = lambda d: (d["resolved_rate"]*100).round(1),
        avg_res_days  = lambda d: d["avg_res_days"].round(1),
    )
)
print("=== CUSTOMER FEEDBACK SUMMARY ===")
print(fb_summary)


=== CUSTOMER FEEDBACK SUMMARY ===
           count  avg_rating  resolved_rate  avg_res_days
sentiment                                                
Negative      74        2.89          57.40         15.10
Neutral       80        2.81          62.90         15.30
Positive     146        2.95          67.90         14.80


### Project 3: Cohort-style Repeat-Customer Analysis

In [275]:
# ── CAPSTONE 3: Identify repeat vs. one-time customers ───────────────────────
# (simulating customer_id derived from sales_rep + region as proxy)
df_c = df_dedup.copy()
df_c["date"] = pd.to_datetime(df_c["date"])

# Proxy customer_id — first letter of rep last name + region
df_c["cust_proxy"] = (
    df_c["sales_rep"].str.split().str[-1].str[0] +
    "_" + df_c["region"]
)

cust_stats = df_c.groupby("cust_proxy").agg(
    first_purchase  = ("date",    "min"),
    last_purchase   = ("date",    "max"),
    num_purchases   = ("revenue", "count"),
    total_spend     = ("revenue", "sum"),
    avg_spend       = ("revenue", "mean"),
).reset_index()

cust_stats["tenure_days"] = (cust_stats["last_purchase"] - cust_stats["first_purchase"]).dt.days
cust_stats["customer_type"] = np.where(cust_stats["num_purchases"] > 1, "Repeat", "One-time")

print("=== CUSTOMER COHORT SUMMARY ===")
print(cust_stats.groupby("customer_type").agg(
    count       = ("cust_proxy","count"),
    avg_spend   = ("total_spend","mean"),
    avg_orders  = ("num_purchases","mean"),
).round(0))


=== CUSTOMER COHORT SUMMARY ===
               count    avg_spend  avg_orders
customer_type                                
Repeat            35 8,389,860.00       14.00


---
## Section 25 — Quick Reference Cheat Sheet

### Reading Data
| Task | Code |
|------|------|
| Read CSV | `pd.read_csv("file.csv")` |
| Read CSV with date parse | `pd.read_csv("file.csv", parse_dates=["date_col"])` |
| Read specific columns | `pd.read_csv("file.csv", usecols=["col1","col2"])` |
| Read Excel, first sheet | `pd.read_excel("file.xlsx")` |
| Read Excel, named sheet | `pd.read_excel("file.xlsx", sheet_name="Sheet1")` |
| Read all sheets | `pd.read_excel("file.xlsx", sheet_name=None)` → dict |
| Read in chunks | `for chunk in pd.read_csv("file.csv", chunksize=1000)` |

### Selecting & Filtering
| Task | Code |
|------|------|
| Select column | `df["col"]` |
| Select multiple columns | `df[["col1","col2"]]` |
| Row by label | `df.loc[0]` |
| Row range by label | `df.loc[0:4, ["col1","col2"]]` |
| Row by position | `df.iloc[0]` |
| Row position range | `df.iloc[0:4, 0:2]` |
| Filter one condition | `df[df["col"] == value]` |
| Filter AND | `df[(df["a"] > x) & (df["b"] == y)]` |
| Filter OR | `df[(df["a"] > x) | (df["b"] == y)]` |
| NOT filter | `df[~(df["col"] == value)]` |
| isin filter | `df[df["col"].isin(["a","b","c"])]` |
| between filter | `df[df["col"].between(10, 50)]` |
| query | `df.query("col > 10 and other == 'x'")` |

### GroupBy & Aggregation
| Task | Code |
|------|------|
| Single agg | `df.groupby("col")["val"].sum()` |
| Multiple aggs | `df.groupby("col")["val"].agg(["sum","mean","count"])` |
| Named aggs | `df.groupby("col").agg(name=("val","sum"))` |
| Multi-column group | `df.groupby(["a","b"])["val"].sum()` |
| Transform (keep row count) | `df.groupby("col")["val"].transform("sum")` |

### Joins
| Type | Code |
|------|------|
| Inner | `pd.merge(df1, df2, on="key", how="inner")` |
| Left | `pd.merge(df1, df2, on="key", how="left")` |
| Right | `pd.merge(df1, df2, on="key", how="right")` |
| Outer | `pd.merge(df1, df2, on="key", how="outer")` |
| Concat rows | `pd.concat([df1, df2], ignore_index=True)` |
| Concat cols | `pd.concat([df1, df2], axis=1)` |

### Missing Values
| Task | Code |
|------|------|
| Detect | `df.isnull().sum()` |
| Drop all-NaN | `df.dropna(how="all")` |
| Drop any-NaN | `df.dropna()` |
| Drop if specific cols NaN | `df.dropna(subset=["col1","col2"])` |
| Fill scalar | `df["col"].fillna(0)` |
| Fill mean | `df["col"].fillna(df["col"].mean())` |
| Forward fill | `df["col"].ffill()` |
| Interpolate | `df["col"].interpolate()` |

### Reshaping
| Task | Code |
|------|------|
| Wide → long | `pd.melt(df, id_vars=["id"], value_vars=["a","b"])` |
| Long → wide | `df.pivot(index="r", columns="c", values="v")` |
| Pivot with agg | `pd.pivot_table(df, values="v", index="r", columns="c", aggfunc="sum")` |
| Stack (cols → rows) | `df.stack()` |
| Unstack (rows → cols) | `df.unstack()` |

### Exporting
| Task | Code |
|------|------|
| To CSV | `df.to_csv("out.csv", index=False)` |
| To Excel (single sheet) | `df.to_excel("out.xlsx", index=False)` |
| To Excel (multi-sheet) | Use `pd.ExcelWriter` context manager |
| To JSON | `df.to_json("out.json", orient="records", indent=2)` |
